# LiteLLM Model class
- Supports calling different model providers through the same API interface. See docs [here](https://docs.litellm.ai/docs/providers)
- Enables cost and token tracking
- Helper classes for APIModelConfig and APIStats. APIModelConfig is used to set model configuration parameters and APIStats is used to track cost and token counts.


In [1]:
import sys, subprocess, importlib.util, os

print("=== Python Executable ===")
print(sys.executable)

print("\n=== sys.path ===")
for p in sys.path:
    print(p)

print("\n=== Installed scikit-learn versions (if any) ===")
subprocess.run(["pip", "show", "scikit-learn"])

print("\n=== Checking if sklearn is importable ===")
spec = importlib.util.find_spec("sklearn")
print("Found sklearn:" if spec else "❌ sklearn not found")

print("\n=== Current Working Directory ===")
print(os.getcwd())


=== Python Executable ===
/data4/anika/UCSBCounterspeechProject/.venv/bin/python

=== sys.path ===
/data4/anika/UCSBCounterspeechProject
/home/anika/.local/share/uv/python/cpython-3.11.14-linux-x86_64-gnu/lib/python311.zip
/home/anika/.local/share/uv/python/cpython-3.11.14-linux-x86_64-gnu/lib/python3.11
/home/anika/.local/share/uv/python/cpython-3.11.14-linux-x86_64-gnu/lib/python3.11/lib-dynload

/data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages

=== Installed scikit-learn versions (if any) ===
Name: scikit-learn
Version: 1.4.1.post1
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License: new BSD
Location: /usr/lib/python3/dist-packages
Requires: 
Required-by: 

=== Checking if sklearn is importable ===
Found sklearn:

=== Current Working Directory ===
/data4/anika/UCSBCounterspeechProject


In [2]:
# Imports

from __future__ import annotations

import logging
from logging import Logger
from typing import Any, Dict, Iterable, Iterator, List, Optional
import json, io
from pathlib import Path

# External libraries
#from dotenv import load_dotenv
import litellm
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_not_exception_type
from pydantic import BaseModel, Field, field_validator
from litellm.types.utils import ModelResponse, Choices
from litellm.exceptions import (
    APIError,
    AuthenticationError,
    BadRequestError,
    ContextWindowExceededError,
    NotFoundError,
    PermissionDeniedError,
)

# Add sklearn for evaluation metrics
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report


In [3]:
# Helper Classes


# Define a MODEL_ALIASES dictionary to easily map long model names to short ones for cli usage. An example is given below.
MODEL_ALIASES = {
    "gpt4o": "gpt-4o",
    "bedrock-claude-3.5-sonnet": "bedrock/arn:aws:bedrock:us-west-2:288380904485:inference-profile/us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    # Add more as needed
}

class APIModelConfig(BaseModel):
    """Configuration for the API model."""

    model_name: str = Field(
        default="gpt4o",
        description="Full litellm model name of the model. See https://docs.litellm.ai/docs/providers",
    )
    temperature: float = Field(
        default=1.0,
        description="The temperature for the model. Default to 1.0 since OpenAI O series does not support other temperature values.",
    )
    top_p: float = Field(default=1.0, description="The top-p value for the model.")
    max_tokens: int = Field(default=1000, description="The maximum number of tokens to generate.")
    host_url: str | None = Field(
        default=None, description="The base URL for the model. Used for some custom model providers such as vLLM/ollama."
    )
    completion_kwargs: dict[str, Any] = Field(
        default={}, description="additional kwargs for the litellm.completion call."
    )

    @field_validator("model_name")
    @classmethod
    def validate_model_name(cls, v: str) -> str:
        """Validate the model name.

        Args:
            v: The model name to validate.

        Returns:
            The validated model name.
        """
        if v in MODEL_ALIASES:
            return MODEL_ALIASES[v]
        return v


# destination: src/models.py
class APIStats(BaseModel):
    """Statistics for the API model."""

    cost: float = 0.0
    tokens_received: int = 0
    tokens_sent: int = 0
    api_calls: int = 0

    def __add__(self, other: APIStats) -> APIStats:
        """Add two APIStats objects together.

        Args:
            other: The other APIStats object to add to the current object.

        Returns:
            A new APIStats object with the sum of the two objects.

        Raises:
            TypeError: If other is not an APIStats object.
        """
        if not isinstance(other, APIStats):
            raise TypeError("APIStats objects can only be added to other APIStats objects")

        return APIStats(**{field: getattr(self, field) + getattr(other, field) for field in self.model_fields})


In [4]:
class LiteLLMModel:
    """A wrapper for the LiteLLM API. Inherit this class to make it a stateful model.

    Cost state management is included in this class, so the inherited classes can implement their own cost management or reuse this class.
    """

    def __init__(self, config: APIModelConfig, name: str, logger: logging.Logger) -> None:
        """Initialize the LiteLLM API Model.

        Args:
            config: The configuration for the LiteLLM API Model
            name: The name of the LiteLLM API Model
            logger: The logger for the LiteLLM API Model
        """
        self.name = name
        self.config = config
        self.stats = APIStats()
        self.logger = logger
        self._setup_client()

    def _setup_client(self) -> None:
        self.model_name = self.config.model_name
        self.model_max_input_tokens = litellm.model_cost.get(self.model_name, {}).get("max_input_tokens")
        self.model_max_output_tokens = litellm.model_cost.get(self.model_name, {}).get("max_output_tokens")
        self.lm_provider = litellm.model_cost.get(self.model_name, {}).get("litellm_provider")
        if self.lm_provider is None or self.config.host_url is not None:
            self.logger.warning(
                f"Using custom host URL: {self.config.host_url}. Cost management will not be available. Register the model with litellm to enable cost management. See https://docs.litellm.ai/docs/completion/token_usage#9-register_model"
            )

    def reset_stats(self, other: APIStats) -> None:
        """Reset or replace the current API Statistics.

        Args:
            other: The other APIStats object to replace the current object with.
        """
        self.stats = other

    def update_stats(self, input_tokens: int, output_tokens: int, cost: float = 0.0) -> None:
        """Update API statistics with new usage information.

        Args:
            input_tokens (int): Number of tokens in the prompt
            output_tokens (int): Number of tokens in the response
            cost (float): Cost of the API call. Defaults to 0.0

        Returns:
            float: The calculated cost of the API call
        """
        self.stats.tokens_sent += input_tokens
        self.stats.tokens_received += output_tokens
        self.stats.cost += cost
        self.stats.api_calls += 1

    @retry(
        wait=wait_random_exponential(min=180, max=360),
        reraise=True,
        stop=stop_after_attempt(3),
        retry=retry_if_not_exception_type(
            (
                RuntimeError,
                NotFoundError,
                PermissionDeniedError,
                ContextWindowExceededError,
                APIError,
                AuthenticationError,
                BadRequestError,
            )
        ),
    )
    def query(
        self,
        messages: list[dict[str, Any]],
        tools: Optional[list[dict[str, Any]]] = None,
    ) -> litellm.types.utils.ModelResponse:
        """Query the model with messages and optional tools.

        Args:
            messages: List of messages in OpenAI format
            tools: Optional list of tool definitions in OpenAI format

        Returns:
            litellm.types.utils.ModelResponse: The complete response from the model
        """
        input_tokens = litellm.utils.token_counter(messages=messages, model=self.model_name)
        extra_args = {}
        if self.config.host_url:
            extra_args["api_base"] = self.config.host_url

        completion_kwargs = self.config.completion_kwargs.copy()
        if self.lm_provider == "anthropic":
            completion_kwargs["max_tokens"] = self.model_max_output_tokens

        # Add tools if provided
        if tools is not None:
            completion_kwargs["tools"] = tools

        try:
            response: litellm.types.utils.ModelResponse = litellm.completion(
                model=self.model_name,
                messages=messages,
                temperature=self.config.temperature,
                top_p=self.config.top_p,
                stream=False,
                drop_params=True,  # Automatically drop unsupported OpenAI params
                **completion_kwargs,
                **extra_args,
            )
        except Exception:
            self.logger.exception(f"Error querying {self.model_name} with tools")
            raise

        choices: litellm.types.utils.Choices = response.choices  # type: ignore
        output_content = choices[0].message.content or ""

        self.logger.debug(f"Input:\n{messages[-1]['content'] if messages else 'No messages'}")
        self.logger.debug(f"Response: {output_content}")

        # Calculate output tokens
        # For tool calls, we need to count tokens in the entire response
        if hasattr(choices[0].message, "tool_calls") and choices[0].message.tool_calls:
            # Include tool calls in token count
            tool_calls_text = str(choices[0].message.tool_calls)
            output_tokens = litellm.utils.token_counter(
                text=f"{output_content}{tool_calls_text}", model=self.model_name
            )
        else:
            output_tokens = litellm.utils.token_counter(text=output_content, model=self.model_name)

        # Update stats
        cost = litellm.cost_calculator.completion_cost(response)
        self.update_stats(input_tokens, output_tokens, cost)

        self.logger.debug(
            f"input_tokens={input_tokens:,}, output_tokens={output_tokens:,}, instance_cost={cost:.2f}, "
            f"total_tokens_sent={self.stats.tokens_sent:,}, total_tokens_received={self.stats.tokens_received:,}, total_cost={self.stats.cost:.2f}, total_api_calls={self.stats.api_calls:,}"
        )

        return response


In [5]:
# === SimplifiedTweets Parser ===
from __future__ import annotations
from pathlib import Path
import json
from typing import List, Dict, Any, Iterable, Optional

class SimplifiedTweetsParser:
    """
    Parser for the attached 50_examples_converted.json format.

    Expected keys per item:
      - ID (int)
      - Username (str)
      - Date (str, e.g. '2021-05-15 02:23:53+00:00')
      - Text (str)
      - Parent_Text (str | None)
      - Media_Photos (List[str])

    Methods:
      - build_user_content(...): returns a single Markdown string (safe for text-only models).
      - build_user_content_parts(...): returns OpenAI-style multimodal parts if you later swap to a vision model.
    """
    
    def __init__(self, json_path: str | Path) -> None:
        self.path = Path(json_path)
        if not self.path.exists():
            raise FileNotFoundError(f"Dataset not found: {self.path}")
        self._data: Optional[List[Dict[str, Any]]] = None

    def load(self) -> List[Dict[str, Any]]:
        if self._data is None:
            with self.path.open("r", encoding="utf-8") as f:
                self._data = json.load(f)
            if not isinstance(self._data, list):
                raise ValueError("Expected a JSON array of records.")
        return self._data

    def _norm_item(self, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "id": item.get("ID"),
            "username": item.get("Username"),
            "date": item.get("Date"),
            "text": item.get("Text") or "",
            "parent_text": item.get("Parent_Text"),
            "images": [u for u in (item.get("Media_Photos") or []) if isinstance(u, str) and u.strip()],
        }

    def iter_items(self) -> Iterable[Dict[str, Any]]:
        for raw in self.load():
            yield self._norm_item(raw)

    @staticmethod
    def _sanitize(s: str, max_len: Optional[int] = None) -> str:
        if s is None:
            return ""
        # keep newlines for readability; strip control chars
        s = s.replace("\r", "")
        if max_len is not None and len(s) > max_len:
            s = s[:max_len - 1] + "…"
        return s

    def build_user_content(
        self,
        *,
        include_parent: bool = True,
        include_images: bool = True,
        max_items: Optional[int] = 200,
        max_total_chars: int = 28000,
        text_truncate: Optional[int] = None,
    ) -> str:
        """
        Returns a single Markdown string suitable for:
          messages = [
            {"role": "system", "content": "..."},
            {"role": "user", "content": user_content},  # <— this field only
          ]

        Safe for text-only models. Image URLs are included inline.
        """
        items = list(self.iter_items())
        total = len(items)

        if max_items is not None:
            items = items[:max_items]

        header = [
            "# Dataset: 50_examples_converted.json",
            f"- Records in file: {total}",
            f"- Records included below: {len(items)}",
            f"- Fields: ID, Username, Date, Text, Parent_Text, Media_Photos",
            "",
            "## Tweets",
        ]

        body_lines: List[str] = []
        char_budget = max_total_chars - sum(len(h) + 1 for h in header)
        for idx, it in enumerate(items, start=1):
            id_ = it["id"]
            username = it["username"]
            date = it["date"]
            text = self._sanitize(it["text"], max_len=text_truncate)
            parent = self._sanitize(it["parent_text"], max_len=text_truncate)
            images = it["images"]

            block_lines = [
                f"### [{idx}] ID={id_}",
                f"- Username: @{username}",
                f"- Date: {date}",
                f"- Text: {text}" if text else "- Text: ",
            ]
            if include_parent:
                block_lines.append(f"- Parent_Text: {parent}" if parent else "- Parent_Text: ")
            if include_images:
                if images:
                    block_lines.append("- Images:")
                    for u in images:
                        # include as both markdown link and image tag for clarity
                        block_lines.append(f"  - {u}")
                        block_lines.append(f"  - ![]({u})")
                else:
                    block_lines.append("- Images: (none)")

            block = "\n".join(block_lines) + "\n"
            if char_budget - len(block) < 0:
                body_lines.append("\n[...truncated to respect max_total_chars...]\n")
                break
            body_lines.append(block)
            char_budget -= len(block)

        return "\n".join(header + body_lines)

    def build_user_content_parts(
        self,
        *,
        include_parent: bool = True,
        max_items: Optional[int] = 50,
        text_truncate: Optional[int] = None,
    ) -> List[Dict[str, Any]]:
        """
        Returns OpenAI-style multimodal content parts:
          [{"type":"text",...}, {"type":"image_url","image_url":{"url": ...}}, ...]

        Use ONLY if your model/endpoint supports image inputs.
        """
        items = list(self.iter_items())
        if max_items is not None:
            items = items[:max_items]

        parts: List[Dict[str, Any]] = [
            {
                "type": "text",
                "text": (
                    "Dataset: 50_examples_converted.json\n"
                    "Each item lists ID, Username, Date, Text, optional Parent_Text, and images.\n"
                    "Below are items with images attached as separate parts.\n"
                ),
            }
        ]
        for idx, it in enumerate(items, start=1):
            text = self._sanitize(it["text"], max_len=text_truncate)
            parent = self._sanitize(it["parent_text"], max_len=text_truncate)
            preface = f"[{idx}] ID={it['id']} | @{it['username']} | {it['date']}\nText: {text}"
            if include_parent:
                preface += f"\nParent_Text: {parent or ''}"
            parts.append({"type": "text", "text": preface})
            for u in it["images"]:
                parts.append({"type": "image_url", "image_url": {"url": u}})
        return parts

# Instantiate and build the content
parser = SimplifiedTweetsParser("50_examples_converted.json")
user_content = parser.build_user_content(
    include_parent=True,
    include_images=True,
    max_items=200,          # adjust as needed
    max_total_chars=28000,  # keep prompts reasonably sized
    text_truncate=None      # or e.g. 500 to clip very long texts
)

# If you later switch to a vision-capable model, you can instead do:
# user_content_parts = parser.build_user_content_parts(include_parent=True, max_items=50)

# Usage
- Create a new file called `.env` with your API key. **VERY IMPORTANT: Make sure this file is in your project's .gitignore**.
- Create a APIModelConfig instance with your model parameters.
- Create a LiteLLMModel instance with APIModelConfig and call the query method.

# Hate Speech Classification

In [6]:
#CELL 1

# Add the openai API key to the .env file
OPENAI_API_KEY="<api_key>"

import logging
import json

# Configure both models
models = {
    "gpt-5-mini": {
        "config": APIModelConfig(model_name="gpt-5-mini", temperature=1, top_p=1.0, max_tokens=1000),
        "model": None
    },
    "gpt-4o": {
        "config": APIModelConfig(model_name="gpt-4o", temperature=1, top_p=1.0, max_tokens=1000),
        "model": None
    }
}

# Initialize models
for model_name, model_info in models.items():
    model_info["model"] = LiteLLMModel(
        config=model_info["config"], 
        name=model_name, 
        logger=logging.getLogger(__name__)
    )

# ============================================================
# PROMPT 1: HATE SPEECH CLASSIFICATION
# ============================================================

hate_speech_system_prompt = """You will classify individual texts (posts or replies) for hate speech. Each text is paired with a unique Twitter/X ID.

Your task: return the labels for each ID using the instructions below. Do not repeat or paraphrase the original text.

Hate Speech Classification

Fields to Add:

1. hate_speech:
- 1 if the text contains hate speech, 0 otherwise.
- Hate speech = any statement that attacks, demeans, dehumanizes, or incites harm against a person or group based on one or more protected characteristics (race, ethnicity, religion, nationality, gender, sexual orientation, disability, etc.).
- Also mark as hate speech if the text endorses, agrees with, or supports a hateful statement, even without adding new content.
- Example: replying “facts,” “so true,” or 👍 to “women are hella dumb” is hate speech because it expresses agreement.

2. hate_speech_type:
- Only assign if hate_speech = 1.

- explicit:
  - Hate speech that is clear, direct, and unambiguous, with no plausible benign interpretation.
  - Key properties:
    - Direct insults, slurs, or profanity aimed at a group
    - Explicit dehumanization (animals, disease, filth, non-human status)
    - Explicit calls for violence, removal, or extermination
    - Meaning is obvious even without context
    - Cannot plausibly be defended as humor, irony, or ambiguity
  - Examples:
    - “Gas all the Jews — it’s the only solution.”
    - “Kill every single one of those goat-fucking refugees.”
    - “Fags aren’t even human. They should be wiped out.”

- implicit:
  - Hate speech that communicates hostility indirectly, relying on context, inference, cultural knowledge, stereotypes, or irony.
  - Key properties:
    - Coded language, euphemisms, or dog whistles
    - Stereotype-based jokes or “humor”
    - Sarcasm or irony masking hostility
    - No explicit call to violence, but still demeaning or dehumanizing
    - Allows plausible deniability
  - Examples:
    - “What do you say to a black guy in a three-piece suit? Will the defendant please rise.”
    - “You want to hear a joke? Women’s rights.”
    - “What’s the only positive thing about Ethiopia? The HIV tests.”

3. identity_targeted:
- Only assign if hate_speech = 1; otherwise use null.
- Choose the identity that is explicitly mentioned or clearly implied in the text.
- Valid options:
  - Women
  - Men
  - Black
  - White
  - Jewish
  - Muslim
  - Middle Eastern
  - Christian
  - Asian
  - Latino/Hispanic
  - LGBTQ+
  - Immigrants/refugees
  - Disabled people
  - Elderly
  - Indigenous
  - Other
  - Unspecified

Critical rules for identity_targeted:
- Choose the group the hate is actually directed at, not a tangential group.
- For replies that express agreement, label the identity targeted in the endorsed statement.
- Re-read before finalizing: did you mark the correct group?

Additional Hate Speech Rules (Must Be Followed):

1. Agreement with Hate Speech:
- If a tweet or reply agrees with hate speech without adding new commentary (eg emojis, “yes exactly,” “facts”), assign:
  - hate_speech = 1
  - the same hate_speech_type and identity_targeted as the endorsed statement.

2. Perspective Test:
- Ask: Would a reasonable member of the targeted group find this message offensive or demeaning?
- If yes, this supports labeling as hate speech.

3. Political Language:
- Political statements alone are not hate speech unless they negatively target a protected group.
- Example:
  - “Pro life more like I hate women” is not hate speech unless it explicitly attacks women as a group.

4. Replying to Third-Party Statements:
- If the text responds to or critiques a quoted or referenced third party, treat the third party’s statement as the parent for hate speech evaluation.

5. Neutral Discussion of Hate Speech:
- If a text neutrally discusses or reports hate speech in external sources (eg news, movies, entertainment, academic discussion), label as not hate speech.
- If the text supports, endorses, or agrees with the hateful content, label as hate speech.

6. Normative Venting Exception:
- Generic expressions of frustration or venting about a group, without violence, dehumanization, stereotypes, or calls for exclusion, should not be labeled as hate speech.
- These statements are typically brief, non-specific, and expressive rather than accusatory or prescriptive.
- Examples (not hate speech):
  - “men suck”
  - “ugh men are trash”
  - “I hate men sometimes”
- If the statement attributes harmful traits, danger, or inferiority to the group, it should be labeled as hate speech.

Output Format:

Return a valid JSON array. Each object must contain exactly these fields:

{
  "id": <ID number>,
  "hate_speech": 0 or 1,
  "hate_speech_type": "explicit" | "implicit" | null,
  "identity_targeted": <group name or null>
}

Return ONLY the JSON array.
No explanations.
No markdown.
No extra text.
"""

# Few-shot examples for hate speech
hate_speech_few_shot = [
# 1 — EXPLICIT, HIGH SEVERITY (anti-Black)
{
"user": """{
"tweet_id": 3001,
"username": "user4410",
"date": "2024-02-11 15:20:10+00:00",
"raw_content": "These streets would be safer if we just beat the shit out of these Black thugs"
}""",
"assistant": """[
{"id": 3001, "hate_speech": 1, "hate_speech_type": "explicit", "hate_speech_severity": "high", "identity_targeted": "Black"}
]"""
},

# 2 — EXPLICIT, HIGH SEVERITY (anti-Jewish)
{
"user": """{
"tweet_id": 3002,
"username": "user8249",
"date": "2024-06-02 09:12:44+00:00",
"raw_content": "we should just burn down every synagogue. That’d fix a lot 😒"
}""",
"assistant": """[
{"id": 3002, "hate_speech": 1, "hate_speech_type": "explicit", "hate_speech_severity": "high", "identity_targeted": "Jewish"}
]"""
},

# 3 — IMPLICIT, HIGH SEVERITY (white supremacist dogwhistle)
{
"user": """{
"tweet_id": 3003,
"username": "user2922",
"date": "2024-01-15 21:05:11+00:00",
"raw_content": "idc what y’all say, we NEED to defend our country from being “replaced.” Wake up."
}""",
"assistant": """[
{"id": 3003, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "high", "identity_targeted": "other/unspecified"}
]"""
},

# 4 — IMPLICIT, MEDIUM (anti-Muslim)
{
"user": """{
"tweet_id": 3004,
"username": "user5091",
"date": "2023-12-10 17:44:02+00:00",
"raw_content": "Whenever something blows up, it’s always the same group 😒 not gonna say who but y’all know."
}""",
"assistant": """[
{"id": 3004, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "medium", "identity_targeted": "Muslim"}
]"""
},

# 5 — IMPLICIT JOKE (anti-women)
{
"user": """{
"tweet_id": 3005,
"username": "user3005",
"date": "2024-03-01 03:10:00+00:00",
"raw_content": "Wanna hear a joke? Women’s rights 😂"
}""",
"assistant": """[
{"id": 3005, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "low", "identity_targeted": "women"}
]"""
},

# 6 — IMPLICIT JOKE (anti-disabled)
{
"user": """{
"tweet_id": 3006,
"username": "user7777",
"date": "2024-04-22 08:55:19+00:00",
"raw_content": "What’s the best part about a vegetable? You can park it anywhere 😭😭"
}""",
"assistant": """[
{"id": 3006, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "low", "identity_targeted": "disabled people"}
]"""
},

# 7 — IMPLICIT STEREOTYPE (anti-men)
{
"user": """{
"tweet_id": 3007,
"username": "user1288",
"date": "2024-01-02 18:33:40+00:00",
"raw_content": "Men are literally useless when it comes to emotions lol."
}""",
"assistant": """[
{"id": 3007, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "low", "identity_targeted": "men"}
]"""
},

# 8 — IMPLICIT GENERALIZATION (anti-Asian)
{
"user": """{
"tweet_id": 3008,
"username": "user1139",
"date": "2024-02-18 11:44:55+00:00",
"raw_content": "Lmao every Asian driver I’ve seen today almost crashed 💀 stay off the roads pls."
}""",
"assistant": """[
{"id": 3008, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "medium", "identity_targeted": "Asian"}
]"""
},

# 9 — IMPLICIT XENOPHOBIA (immigrants)
{
"user": """{
"tweet_id": 3009,
"username": "user7410",
"date": "2024-05-10 01:10:07+00:00",
"raw_content": "Not tryna be mean but these immigrants don’t even TRY to fit in."
}""",
"assistant": """[
{"id": 3009, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "low", "identity_targeted": "immigrants/refugees"}
]"""
},

# 10 — EXPLICIT HARASSMENT (anti-LGBTQ+)
{
"user": """{
"tweet_id": 3010,
"username": "user5881",
"date": "2024-01-19 14:02:32+00:00",
"raw_content": "These gay dudes make me sick bro 🤢 keep that shit away from me."
}""",
"assistant": """[
{"id": 3010, "hate_speech": 1, "hate_speech_type": "explicit", "hate_speech_severity": "medium", "identity_targeted": "LGBTQ+"}
]"""
},

# 11 — ENDORSEMENT OF PARENT HATE
{
"user": """{
"tweet_id": 3011,
"username": "user9040",
"date": "2024-03-11 04:11:44+00:00",
"raw_content": "facts lmao they really ARE all criminals"
}""",
"assistant": """[
{"id": 3011, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "medium", "identity_targeted": "immigrants/refugees"}
]"""
},

# 12 — IMPLICIT ANTI-JEWISH JOKE
{
"user": """{
"tweet_id": 3012,
"username": "user3100",
"date": "2024-04-24 06:12:56+00:00",
"raw_content": "What’s the difference between a Jew and a pizza? A pizza doesn’t scream in the oven 💀"
}""",
"assistant": """[
{"id": 3012, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "high", "identity_targeted": "Jewish"}
]"""
},

# 13 — NON-HATE (sounds negative but not identity-based)
{
"user": """{
"tweet_id": 3013,
"username": "user8005",
"date": "2024-01-30 16:40:00+00:00",
"raw_content": "I hate fake people so much oml."
}""",
"assistant": """[
{"id": 3013, "hate_speech": 0, "hate_speech_type": null, "hate_speech_severity": null, "identity_targeted": null}
]"""
},

# 14 — NON-HATE (swearing but no identity)
{
"user": """{
"tweet_id": 3014,
"username": "user2233",
"date": "2024-01-25 08:22:11+00:00",
"raw_content": "Bro you’re a fucking idiot 😂 but go off ig"
}""",
"assistant": """[
{"id": 3014, "hate_speech": 0, "hate_speech_type": null, "hate_speech_severity": null, "identity_targeted": null}
]"""
},

# 15 — NON-HATE (anger at specific person, not group)
{
"user": """{
"tweet_id": 3015,
"username": "user0320",
"date": "2024-04-11 11:40:29+00:00",
"raw_content": "@User shut up dude you’re clueless"
}""",
"assistant": """[
{"id": 3015, "hate_speech": 0, "hate_speech_type": null, "hate_speech_severity": null, "identity_targeted": null}
]"""
},

# 16 — IMPLICIT ANTI-CHRISTIAN
{
"user": """{
"tweet_id": 3016,
"username": "user2291",
"date": "2024-05-13 19:45:12+00:00",
"raw_content": "Christians always forcing their beliefs on everyone 🙄 literally the worst."
}""",
"assistant": """[
{"id": 3016, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "medium", "identity_targeted": "Christian"}
]"""
},

# 17 — IMPLICIT ANTI-LATINO
{
"user": """{
"tweet_id": 3017,
"username": "user6661",
"date": "2024-03-15 20:50:33+00:00",
"raw_content": "Bro every Latino dude at this bar is trying to fight 💀 chill pls."
}""",
"assistant": """[
{"id": 3017, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "medium", "identity_targeted": "Latino/Hispanic"}
]"""
},

# 18 — EXPLICIT, LOW (anti-women, insult)
{
"user": """{
"tweet_id": 3018,
"username": "user5579",
"date": "2024-02-08 02:15:19+00:00",
"raw_content": "Women are so damn stupid sometimes omg."
}""",
"assistant": """[
{"id": 3018, "hate_speech": 1, "hate_speech_type": "explicit", "hate_speech_severity": "low", "identity_targeted": "women"}
]"""
},

# 19 — IMPLICIT, LOW (anti-elderly)
{
"user": """{
"tweet_id": 3019,
"username": "user4044",
"date": "2024-04-01 12:11:07+00:00",
"raw_content": "Old people should NOT be allowed to drive anymore fr."
}""",
"assistant": """[
{"id": 3019, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "low", "identity_targeted": "other/unspecified"}
]"""
},

# 20 — NON-HATE (positive)
{
"user": """{
"tweet_id": 3020,
"username": "user1293",
"date": "2024-02-20 09:10:55+00:00",
"raw_content": "love my gay friends sm they always hype me up lol"
}""",
"assistant": """[
{"id": 3020, "hate_speech": 0, "hate_speech_type": null, "hate_speech_severity": null, "identity_targeted": null}
]"""
}
]

# ============================================================
# PROMPT 2: COUNTERSPEECH CLASSIFICATION
# ============================================================

counterspeech_system_prompt = """You will classify individual texts (replies) for counterspeech. Each text is paired with a unique Twitter/X ID and includes the parent text it's replying to.
Your job: return the counterspeech labels for each ID using the instructions below. Do not repeat the original text.

Counterspeech Classification
Fields to Add:

1. counterspeech:
   * 1 if the text challenges, refutes, condemns, or even mildly disagrees with hateful, discriminatory, or harmful content. 0 if otherwise.
   * Be liberal in detection: Even subtle disagreement, questioning, or redirection counts as counterspeech.
   * Look for: corrections, questioning premises, expressing discomfort, humor/sarcasm undermining hate, warning of consequences, expressing support for targets, empathetic language, or use hostile language to counter hate.
   * DO NOT require explicit condemnation - even mild pushback counts.
   * If the parent text contains hate speech, discriminatory views, or harmful stereotypes, and the reply challenges it, mark as counterspeech.

2. counterspeech_type:
   * If counterspeech = 1, choose the relevant categories out of the following (else null).

Category | Definition | Example
'presenting facts' | Correcting false or misleading statements with evidence. | 'Actually, refugee crime rates are lower than native-born citizens.'
'pointing out hypocrisy or contradictions' | Highlighting inconsistencies in the hate speaker's logic. Asking counter-questions. | 'You preach family values but cheat on your spouse — maybe rethink judging others.'
'warning of consequences' | Pointing out possible legal, social, or personal repercussions. | 'Inciting violence online can get you charged with a felony.'
'affiliation' | Establishing shared identity or solidarity with the target or both groups. | 'I'm a veteran, and I fought for everyone's freedom — including Muslims.'
'denouncing hate speech' | Explicitly condemning the hateful message. | 'This is blatant racism and it's disgusting.'
'humor' | Using jokes, irony, or sarcasm to undermine hate. | 'Wow, you've unlocked the 'Medieval Thinking' achievement badge.'
'empathy/positive tone' | Expressing compassion, kindness, or unity. | 'We're all trying to live decent lives — hate helps no one.'
'hostile language' | Responding to hate with insults or profanity while countering the message. | 'Only an idiot would believe that garbage.'

3. dominant_counterspeech_type:
   * If there is no counterspeech present, put null.
   * If there is one type of counterspeech present, put that type.
   * If there are multiple types present, choose the most salient and emphasized form.

IMPORTANT: Pay special attention to parent_hate_speech=1 cases. If the parent contains hate speech and the reply disagrees, questions, or challenges it in ANY way, mark as counterspeech=1.

Output Format:
Return your response as a valid JSON array containing objects for each ID. Each object must have these exact fields:

'id': (the ID number)
'counterspeech': (0 or 1)
'counterspeech_type': (one of the types listed above or null)
'dominant_counterspeech_type': (same as counterspeech_type if only one, or the dominant one if multiple, or null)

Return ONLY the JSON array, no other text, no markdown formatting."""

# Few-shot examples for counterspeech
counterspeech_few_shot = [
    {
        "user": """{
"tweet_id": 2001,
"username": "user0842",
"date": "2024-06-02 11:12:00+00:00",
"raw_content": "@HotTakes That’s just false. FBI reports show immigrant crime rates are lower than native-born citizens.",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2001, "counterspeech": 1, "counterspeech_type": "presenting facts", "dominant_counterspeech_type": "presenting facts"}
]"""
    },
    {
        "user": """{
"tweet_id": 2002,
"username": "user4921",
"date": "2024-06-02 11:15:44+00:00",
"raw_content": "@HotTakes So you’re mad at immigrants for 'stealing jobs' AND for not working? Pick one.",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2002, "counterspeech": 1, "counterspeech_type": "pointing out hypocrisy or contradictions", "dominant_counterspeech_type": "pointing out hypocrisy or contradictions"}
]"""
    },
    {
    "user": """{
"tweet_id": 2020,
"username": "user1130",
"date": "2024-06-02 12:20:33+00:00",
"raw_content": "how does blaming millions of people you’ve never met solve anything?",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
    "assistant": """[
{"id": 2020, "counterspeech": 1, "counterspeech_type": "pointing out hypocrisy or contradictions", "dominant_counterspeech_type": "pointing out hypocrisy or contradictions"}
]"""
    },
    {
        "user": """{
"tweet_id": 2003,
"username": "user7754",
"date": "2024-06-02 11:17:02+00:00",
"raw_content": "@HotTakes Careful with posts like this, people get suspended for spreading hate.",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2003, "counterspeech": 1, "counterspeech_type": "warning of consequences", "dominant_counterspeech_type": "warning of consequences"}
]"""
    },
    {
        "user": """{
"tweet_id": 2004,
"username": "user9350",
"date": "2024-06-02 11:19:30+00:00",
"raw_content": "@HotTakes These comments just hurt people who are trying to survive. Everyone deserves a chance at safety.",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2004, "counterspeech": 1, "counterspeech_type": "empathy/positive tone", "dominant_counterspeech_type": "empathy/positive tone"}
]"""
    },
    {
        "user": """{
"tweet_id": 2005,
"username": "user2409",
"date": "2024-06-02 11:20:55+00:00",
"raw_content": "@HotTakes I served with immigrants in the Army. They fought harder than most of us. Show some respect.",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2005, "counterspeech": 1, "counterspeech_type": "affiliation", "dominant_counterspeech_type": "affiliation"}
]"""
    },
    {
        "user": """{
"tweet_id": 2006,
"username": "user5832",
"date": "2024-06-02 11:23:10+00:00",
"raw_content": "@HotTakes Ah yes, every problem in the world traced back to immigrants. Very original take.",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2006, "counterspeech": 1, "counterspeech_type": "humor", "dominant_counterspeech_type": "humor"}
]"""
    },
    {
        "user": """{
"tweet_id": 2007,
"username": "user7731",
"date": "2024-06-02 11:24:41+00:00",
"raw_content": "@HotTakes This is racist garbage. Stop posting this trash.",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2007, "counterspeech": 1, "counterspeech_type": "denouncing hate speech", "dominant_counterspeech_type": "denouncing hate speech"}
]"""
    },
    {
        "user": """{
"tweet_id": 2008,
"username": "user6024",
"date": "2024-06-02 11:25:59+00:00",
"raw_content": "@HotTakes You sound like an idiot man stfu",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2008, "counterspeech": 1, "counterspeech_type": "hostile language", "dominant_counterspeech_type": "hostile language"}
]"""
    },
    {
        "user": """{
"tweet_id": 2010,
"username": "user3441",
"date": "2024-06-02 11:32:50+00:00",
"raw_content": "@HotTakes bro did you see the Lakers choke again 💀",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2010, "counterspeech": 0, "counterspeech_type": null, "dominant_counterspeech_type": null}
]"""
    },
    {
        "user": """{
"tweet_id": 2014,
"username": "user0064",
"date": "2024-06-02 11:50:33+00:00",
"raw_content": "@HotTakes facts bro say it louder",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2014, "counterspeech": 0, "counterspeech_type": null, "dominant_counterspeech_type": null}
]"""
    },
    {
        "user": """{
"tweet_id": 2015,
"username": "user5622",
"date": "2024-06-02 11:53:10+00:00",
"raw_content": "@HotTakes I get being frustrated but blaming a whole group doesn't help anything.",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2015, "counterspeech": 1, "counterspeech_type": "empathy/positive tone", "dominant_counterspeech_type": "empathy/positive tone"}
]"""
    },
    {
        "user": """{
"tweet_id": 2018,
"username": "user8020",
"date": "2024-06-02 12:10:44+00:00",
"raw_content": "@HotTakes lol ok dude",
"parent_tweet_id": 2000,
"parent_raw_content": "Immigrants commit all the crimes. It's obvious.",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 2018, "counterspeech": 0, "counterspeech_type": null, "dominant_counterspeech_type": null}
]"""
    }
]

# ============================================================
# Helper functions
# ============================================================

def parse_model_output(raw_text):
    """Safely parse model JSON output into Python list."""
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        try:
            start = raw_text.find("[")
            end = raw_text.rfind("]") + 1
            return json.loads(raw_text[start:end])
        except Exception:
            return []

def classify_two_stage(batch, model_name):
    """
    Run hate speech classification first, then counterspeech classification.
    Returns combined predictions.
    Handles flexible field names for compatibility.
    
    IMPROVED: Now properly links parent-reply relationships and passes parent hate speech labels.
    """
    model = models[model_name]["model"]
    
    # Helper to normalize field names
    def normalize_item(item):
        """Convert various field name formats to standard format"""
        normalized = {}
        
        # ID field
        normalized['tweet_id'] = item.get('tweet_id') or item.get('id')
        
        # Username field
        normalized['username'] = item.get('username') or item.get('Username') or 'unknown'
        
        # Date field
        normalized['date'] = item.get('date') or item.get('Date') or 'unknown'
        
        # Content field
        normalized['raw_content'] = (
            item.get('raw_content') or 
            item.get('text') or 
            item.get('Text') or 
            item.get('content') or 
            ''
        )
        
        # Parent fields
        normalized['parent_tweet_id'] = (
            item.get('parent_tweet_id') or 
            item.get('parent_id') or 
            None
        )
        normalized['parent_raw_content'] = (
            item.get('parent_raw_content') or 
            item.get('parent_text') or 
            item.get('Parent_Text') or 
            ''
        )
        
        return normalized
    
    # Normalize all items in batch
    normalized_batch = [normalize_item(item) for item in batch]
    
    # Stage 1: Hate Speech Classification for ALL posts (including parent texts)
    # We need to classify both the posts AND their parents
    hate_messages = [
        {"role": "system", "content": hate_speech_system_prompt}
    ]
    for ex in hate_speech_few_shot:
        hate_messages.append({"role": "user", "content": ex["user"]})
        hate_messages.append({"role": "assistant", "content": ex["assistant"]})
    
    # Build list of texts to classify: both posts and their unique parents
    texts_to_classify = []
    parent_texts_map = {}  # Map parent text -> synthetic ID for classification
    
    for ex in normalized_batch:
        # Add the post itself
        texts_to_classify.append({
            "tweet_id": ex['tweet_id'],
            "username": ex['username'],
            "date": ex['date'],
            "raw_content": ex['raw_content']
        })
        
        # If it has a parent, add parent for classification too
        parent_text = ex.get('parent_raw_content', '').strip()
        if parent_text and parent_text not in parent_texts_map:
            # Create synthetic ID for parent (negative of post ID to avoid collisions)
            parent_id = -(ex['tweet_id'])
            parent_texts_map[parent_text] = parent_id
            texts_to_classify.append({
                "tweet_id": parent_id,
                "username": "parent_unknown",
                "date": ex['date'],
                "raw_content": parent_text
            })
    
    # Create hate speech classification prompt
    hate_items = []
    for text_entry in texts_to_classify:
        raw_content = (text_entry.get('raw_content') or '').replace('"', "'").replace('\n', ' ')
        hate_items.append({
            "tweet_id": text_entry['tweet_id'],
            "username": text_entry['username'],
            "date": text_entry['date'],
            "raw_content": raw_content
        })
    
    hate_user_content = json.dumps(hate_items, indent=2, ensure_ascii=False)
    hate_messages.append({"role": "user", "content": hate_user_content})
    
    # Query hate speech
    hate_response = model.query(messages=hate_messages)
    hate_raw = hate_response.choices[0].message.content
    hate_preds = parse_model_output(hate_raw)
    
    # Create lookup for hate speech results
    hate_lookup = {item["id"]: item for item in hate_preds if "id" in item}
    
    # Reverse map: parent text -> hate speech classification
    parent_hate_lookup = {}
    for parent_text, parent_id in parent_texts_map.items():
        if parent_id in hate_lookup:
            parent_hate_lookup[parent_text] = hate_lookup[parent_id]
    
    # Stage 2: Counterspeech Classification
    counter_messages = [
        {"role": "system", "content": counterspeech_system_prompt}
    ]
    for ex in counterspeech_few_shot:
        counter_messages.append({"role": "user", "content": ex["user"]})
        counter_messages.append({"role": "assistant", "content": ex["assistant"]})
    
    # Build counterspeech prompt with parent hate speech labels
    counter_items = []
    for ex in normalized_batch:
        parent_content = (ex.get('parent_raw_content') or '').replace('"', "'").replace('\n', ' ')
        raw_content = (ex.get('raw_content') or '').replace('"', "'").replace('\n', ' ')
        
        # Get parent hate speech classification if available
        parent_hate = parent_hate_lookup.get(ex.get('parent_raw_content', '').strip(), {})
        parent_hate_speech = parent_hate.get('hate_speech', 0)
        parent_hate_type = parent_hate.get('hate_speech_type', None)
        
        counter_item = {
            "tweet_id": ex['tweet_id'],
            "username": ex['username'],
            "date": ex['date'],
            "raw_content": raw_content,
            "parent_tweet_id": ex.get('parent_tweet_id'),
            "parent_raw_content": parent_content,
            "parent_hate_speech": parent_hate_speech,
            "parent_hate_speech_type": parent_hate_type
        }
        counter_items.append(counter_item)
    
    # Convert to JSON string
    counter_user_content = json.dumps(counter_items, indent=2, ensure_ascii=False)
    counter_messages.append({"role": "user", "content": counter_user_content})
    
    # Query counterspeech
    counter_response = model.query(messages=counter_messages)
    counter_raw = counter_response.choices[0].message.content
    counter_preds = parse_model_output(counter_raw)
    
    # Merge predictions (only include actual posts, not synthetic parent entries)
    merged_preds = {}
    for item in hate_preds:
        if "id" in item and item["id"] > 0:  # Only real posts (positive IDs)
            merged_preds[item["id"]] = item
    
    for item in counter_preds:
        if "id" in item:
            if item["id"] in merged_preds:
                merged_preds[item["id"]].update(item)
            else:
                merged_preds[item["id"]] = item
    
    return merged_preds, hate_raw, counter_raw


# Example usage for single test
if __name__ == "__main__":
    # Test with a small example using realistic Twitter data structure
    test_batch = [
        {
            "tweet_id": 1823456712390884356,
            "username": "RealTruthTeller",
            "date": "2023-06-10 18:24:55+00:00",
            "raw_content": "Immigrants are ruining our country. They take jobs, cause crime, and drain our economy. #facts",
            "parent_tweet_id": None,
            "parent_raw_content": None
        },
        {
            "tweet_id": 1823460021399814723,
            "username": "CivicMind",
            "date": "2023-06-10 18:37:19+00:00",
            "raw_content": "@RealTruthTeller That's just not true. Immigrants pay taxes, start small businesses, and make communities stronger.",
            "parent_tweet_id": 1823456712390884356,
            "parent_raw_content": "Immigrants are ruining our country. They take jobs, cause crime, and drain our economy. #facts"
        }
    ]
    
    print("\n=== Testing with gpt-5-mini ===")
    preds_mini, hate_raw_mini, counter_raw_mini = classify_two_stage(test_batch, "gpt-5-mini")
    print("\nHate Speech Output:")
    print(hate_raw_mini)
    print("\nCounterspeech Output:")
    print(counter_raw_mini)
    print("\nMerged Predictions:")
    print(json.dumps(preds_mini, indent=2))
    
    print("\n=== Testing with gpt-4o ===")
    preds_4o, hate_raw_4o, counter_raw_4o = classify_two_stage(test_batch, "gpt-4o")
    print("\nHate Speech Output:")
    print(hate_raw_4o)
    print("\nCounterspeech Output:")
    print(counter_raw_4o)
    print("\nMerged Predictions:")
    print(json.dumps(preds_4o, indent=2))




=== Testing with gpt-5-mini ===

Hate Speech Output:
[
  {
    "id": 1823456712390884356,
    "hate_speech": 1,
    "hate_speech_type": "implicit",
    "identity_targeted": "Immigrants/refugees"
  },
  {
    "id": 1823460021399814723,
    "hate_speech": 0,
    "hate_speech_type": null,
    "identity_targeted": null
  },
  {
    "id": -1823460021399814723,
    "hate_speech": 1,
    "hate_speech_type": "implicit",
    "identity_targeted": "Immigrants/refugees"
  }
]

Counterspeech Output:
[
  {
    "id": 1823456712390884356,
    "counterspeech": 0,
    "counterspeech_type": null,
    "dominant_counterspeech_type": null
  },
  {
    "id": 1823460021399814723,
    "counterspeech": 1,
    "counterspeech_type": "presenting facts",
    "dominant_counterspeech_type": "presenting facts"
  }
]

Merged Predictions:
{
  "1823456712390884356": {
    "id": 1823456712390884356,
    "hate_speech": 1,
    "hate_speech_type": "implicit",
    "identity_targeted": "Immigrants/refugees",
    "counterspeec

In [7]:


#CELL 2

import time, json, numpy as np
from statistics import mean
from sklearn.metrics import accuracy_score, f1_score

# --- Load and merge data with labels ---
# Load the text data
with open("50_examples_converted.json", "r", encoding="utf-8") as f:
    text_data = {int(d["ID"]): d for d in json.load(f)}

# Load the labels
with open("50_annotated_examples.json", "r", encoding="utf-8") as f:
    labels = {int(d["id"]): d for d in json.load(f)}

# Merge text data with labels to create complete ground truth
ground_truth = {}
for id_val in labels.keys():
    if id_val in text_data:
        # Merge both dictionaries
        ground_truth[id_val] = {**text_data[id_val], **labels[id_val]}
    else:
        print(f"Warning: ID {id_val} has labels but no text data")

print(f"Loaded {len(ground_truth)} examples with text and labels")

# --- Prepare data for classification ---
# Convert ground truth to list format for classification
examples = []
for id_val, data in ground_truth.items():
    example = {
        'tweet_id': data.get('ID'),
        'id': data.get('ID'),  # Keep both for compatibility
        'username': data.get('Username', 'unknown'),
        'date': data.get('Date', 'unknown'),
        'raw_content': data.get('Text', ''),
        'text': data.get('Text', ''),  # Keep both for compatibility
        'parent_raw_content': data.get('Parent_Text'),
        'parent_text': data.get('Parent_Text'),  # Keep both for compatibility
        'parent_tweet_id': None  # Will be set if we can find parent in dataset
    }
    examples.append(example)

print(f"Prepared {len(examples)} examples for classification")
print(f"Sample example keys: {list(examples[0].keys())}")

# Remaining code continues as before
batch_sizes = [50]
num_iterations = 1
model_names = ["gpt-5-mini", "gpt-4o"]
results_summary = []

# ==============================================================
# Helper functions
# ==============================================================

def parse_model_output(raw_text):
    """Safely parse model JSON output into Python list."""
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        try:
            start = raw_text.find("[")
            end = raw_text.rfind("]") + 1
            return json.loads(raw_text[start:end])
        except Exception:
            return []

# ==============================================================
# Metric computation
# ==============================================================

def compute_metrics(preds, gold):
    """Compute metrics for binary, multiclass, and multilabel fields."""

    # --- Helper normalizers ---
    def safe_label(val):
        if not val or str(val).lower() in {"unknown", "none", "null", "nan"}:
            return "none"
        return str(val).strip()

    def normalize_identities(x):
        if x is None:
            return []
        if isinstance(x, str):
            return [x]
        if isinstance(x, list):
            return x
        return []

    # --- Binary fields ---
    y_true_hate, y_pred_hate = [], []
    y_true_counter, y_pred_counter = [], []

    # --- Multiclass fields ---
    y_true_type, y_pred_type = [], []
    y_true_severity, y_pred_severity = [], []
    y_true_counter_type, y_pred_counter_type = [], []

    # --- Multilabel (identities) ---
    base_identities = [
        'women', 'men', 'Black', 'White', 'Jewish', 'Muslim', 'Christian',
        'Asian', 'Latino/Hispanic', 'LGBTQ+', 'immigrants/refugees', "elderly",
        'disabled people', 'Indigenous', 'middle eastern', 'other', 'unspecified'
    ]
    all_identities = set(base_identities)

    # Dynamically extend identity list with unseen ones
    for d in gold.values():
        for x in normalize_identities(d.get("identity_targeted")):
            all_identities.add(x)
    for p in preds.values():
        for x in normalize_identities(p.get("identity_targeted")):
            all_identities.add(x)
    all_identities = sorted(all_identities)

    y_true_id, y_pred_id = [], []

    # --- Collect gold and predicted values ---
    for pid, p in preds.items():
        g = gold.get(pid)
        if not g:
            continue

        # Binary
        y_true_hate.append(g["hate_speech"])
        y_pred_hate.append(p.get("hate_speech", 0))
        y_true_counter.append(g["counterspeech"])
        y_pred_counter.append(p.get("counterspeech", 0))

        # Multiclass (cleaned)
        y_true_type.append(safe_label(g.get("hate_speech_type")))
        y_pred_type.append(safe_label(p.get("hate_speech_type")))
        y_true_severity.append(safe_label(g.get("hate_speech_severity")))
        y_pred_severity.append(safe_label(p.get("hate_speech_severity")))
        y_true_counter_type.append(safe_label(g.get("counterspeech_type")))
        y_pred_counter_type.append(safe_label(p.get("counterspeech_type")))

        # Multilabel (normalized)
        gt_raw = normalize_identities(g.get("identity_targeted"))
        pr_raw = normalize_identities(p.get("identity_targeted"))
        gt_set, pr_set = set(gt_raw), set(pr_raw)
        y_true_id.append([1 if i in gt_set else 0 for i in all_identities])
        y_pred_id.append([1 if i in pr_set else 0 for i in all_identities])

    # --- Compute metrics safely ---
    metrics = {
        # Binary
        "hate_acc": round(accuracy_score(y_true_hate, y_pred_hate), 3),
        "hate_f1": round(f1_score(y_true_hate, y_pred_hate), 3),
        "counter_acc": round(accuracy_score(y_true_counter, y_pred_counter), 3),
        "counter_f1": round(f1_score(y_true_counter, y_pred_counter), 3),

        # Multiclass (macro F1 averages over all categories)
        "type_acc": round(accuracy_score(y_true_type, y_pred_type), 3),
        "type_f1": round(f1_score(y_true_type, y_pred_type, average="macro"), 3),

        "severity_acc": round(accuracy_score(y_true_severity, y_pred_severity), 3),
        "severity_f1": round(f1_score(y_true_severity, y_pred_severity, average="macro"), 3),

        "counter_type_acc": round(accuracy_score(y_true_counter_type, y_pred_counter_type), 3),
        "counter_type_f1": round(f1_score(y_true_counter_type, y_pred_counter_type, average="macro"), 3),

        # Multilabel (micro average)
        "identity_acc": round(accuracy_score(np.array(y_true_id).flatten(),
                                             np.array(y_pred_id).flatten()), 3),
        "identity_f1": round(f1_score(y_true_id, y_pred_id, average="micro"), 3)
    }

    return metrics

# ==============================================================
# Run experiments comparing both models
# ==============================================================

for model_name in model_names:
    print(f"\n{'='*60}")
    print(f"Testing model: {model_name}")
    print(f"{'='*60}")
    
    for batch_size in batch_sizes:
        print(f"\n=== Testing batch size {batch_size} for {num_iterations} iterations ===")
        all_iter_results = []
        all_iter_times = []

        for iteration in range(num_iterations):
            print(f"Iteration {iteration + 1}/{num_iterations}")
            batch_times, preds = [], {}

            for i in range(0, len(examples), batch_size):
                batch = examples[i:i + batch_size]
                start = time.time()
                merged_preds, _, _ = classify_two_stage(batch, model_name)
                elapsed = time.time() - start
                batch_times.append(elapsed)
                preds.update(merged_preds)

                # ✅ Pydantic v2 compatible model stats logging
                with open(f"usage_log_{model_name}.json", "w") as f:
                    json.dump(models[model_name]["model"].stats.model_dump(), f, indent=2)
    
            # --- Record metrics for this iteration ---
            metrics = compute_metrics(preds, ground_truth)
            total_time, avg_time = sum(batch_times), mean(batch_times)
            all_iter_times.append(avg_time)
            all_iter_results.append(metrics)

        # --- Aggregate over iterations ---
        avg_metrics = {
            "hate_acc": round(mean([m["hate_acc"] for m in all_iter_results]), 3),
            "hate_f1": round(mean([m["hate_f1"] for m in all_iter_results]), 3),
            "counter_acc": round(mean([m["counter_acc"] for m in all_iter_results]), 3),
            "counter_f1": round(mean([m["counter_f1"] for m in all_iter_results]), 3),
            "type_f1": round(mean([m["type_f1"] for m in all_iter_results]), 3),
            "severity_f1": round(mean([m["severity_f1"] for m in all_iter_results]), 3),
            "counter_type_f1": round(mean([m["counter_type_f1"] for m in all_iter_results]), 3),
            "identity_f1": round(mean([m["identity_f1"] for m in all_iter_results]), 3),
        }

        results_summary.append({
            "model": model_name,
            "batch_size": batch_size,
            "iterations": num_iterations,
            "avg_time_s": round(mean(all_iter_times), 2),
            **avg_metrics,
        })

# ==============================================================
# Final Summary with Model Comparison
# ==============================================================

print("\n" + "="*80)
print("FINAL COMPARISON: GPT-5-mini vs GPT-4o")
print("="*80)
for r in results_summary:
    print(f"\nModel: {r['model']}")
    print(f"  Batch Size: {r['batch_size']}")
    print(f"  Avg Time: {r['avg_time_s']}s")
    print(f"  Hate Speech - Acc: {r['hate_acc']}, F1: {r['hate_f1']}")
    print(f"  Counterspeech - Acc: {r['counter_acc']}, F1: {r['counter_f1']}")
    print(f"  Type F1: {r['type_f1']}, Severity F1: {r['severity_f1']}")
    print(f"  Counter Type F1: {r['counter_type_f1']}, Identity F1: {r['identity_f1']}")


Loaded 50 examples with text and labels
Prepared 50 examples for classification
Sample example keys: ['tweet_id', 'id', 'username', 'date', 'raw_content', 'text', 'parent_raw_content', 'parent_text', 'parent_tweet_id']

Testing model: gpt-5-mini

=== Testing batch size 50 for 1 iterations ===
Iteration 1/1

Testing model: gpt-4o

=== Testing batch size 50 for 1 iterations ===
Iteration 1/1

FINAL COMPARISON: GPT-5-mini vs GPT-4o

Model: gpt-5-mini
  Batch Size: 50
  Avg Time: 235.34s
  Hate Speech - Acc: 0.84, F1: 0.6
  Counterspeech - Acc: 0.88, F1: 0.4
  Type F1: 0.578, Severity F1: 1.0
  Counter Type F1: 0.229, Identity F1: 0.6

Model: gpt-4o
  Batch Size: 50
  Avg Time: 25.18s
  Hate Speech - Acc: 0.78, F1: 0.645
  Counterspeech - Acc: 0.8, F1: 0.0
  Type F1: 0.476, Severity F1: 1.0
  Counter Type F1: 0.111, Identity F1: 0.387


In [ ]:
# 
#
#
#      PREVIOUS VERSION
#
#
#

#CELL 1

# Add the openai API key to the .env file
OPENAI_API_KEY="<api_key>"

import logging
import json

# Configure both models
models = {
    "gpt-5-mini": {
        "config": APIModelConfig(model_name="gpt-5-mini", temperature=1, top_p=1.0, max_tokens=1000),
        "model": None
    },
    "gpt-4o": {
        "config": APIModelConfig(model_name="gpt-4o", temperature=1, top_p=1.0, max_tokens=1000),
        "model": None
    }
}

# Initialize models
for model_name, model_info in models.items():
    model_info["model"] = LiteLLMModel(
        config=model_info["config"], 
        name=model_name, 
        logger=logging.getLogger(__name__)
    )

# ============================================================
# PROMPT 1: HATE SPEECH CLASSIFICATION
# ============================================================

hate_speech_system_prompt = """You will classify individual texts (posts or replies) for hate speech. Each text is paired with a unique Twitter/X ID.
Your job: return the labels for each ID using the instructions below. Do not repeat the original text.

Hate Speech Classification
Fields to Add:

1. hate_speech:
   * 1 if the text contains hate speech, 0 otherwise.
   * Hate speech = any statement that attacks, demeans, or incites harm against a person or group based on one or more protected characteristics (race, ethnicity, religion, nationality, gender, sexual orientation, disability, etc.).
   * Also mark as hate speech if the text endorses, supports, or agrees with a hateful statement.
     * For example, if a post says 'facts' or 'so true' in response to 'women are hella dumb', it should be labeled as hate speech because it expresses agreement with a hateful sentiment.

2. hate_speech_type:
   * 'explicit' → overt, direct use of slurs, severe derogatory language, or clear calls for violence.
     * Examples:
       * 'Gas all the Jews — it's the only solution.' (call for genocide)
       * 'Kill every single one of those goat-fucking refugees, they don't deserve to live.' (incitement to mass violence)
       * 'Lynch that nigger before he infects the neighborhood.' (explicit racial violence)
       * 'Fags aren't even human. they should be wiped out.' (dehumanization + extermination)
   * 'implicit' → indirect, coded, sarcastic, or requiring context to detect — often euphemisms, plausible deniability, or dog whistles.
     * Examples:
       * 'What do you say to a black guy in a three piece suit? Will the defendant please rise.'
       * 'What part of a vegetable can't you eat? The wheelchair.'
       * 'You want to hear a joke? Women's rights.'
       * 'Why should you never marry a Jewish woman? When dinner time comes, they're too scared to get near the oven.'
       * 'What's the only positive thing about Ethiopia? The HIV tests.'

3. hate_speech_severity:
   * 'low' → Stereotypes, generalizations, or mildly demeaning remarks with low likelihood of direct harm.
     * Examples:
       * 'Women are too emotional to lead.'
       * 'Old people can't understand technology.'
   * 'medium' → Targeted insults, harassment, or demeaning language likely to reinforce discrimination but without direct calls for violence.
     * Examples:
       * 'Send all the immigrants home, they don't belong here.'
       * 'Gays are disgusting and ruin the moral fabric of society.'
   * 'high' → Explicit calls for violence, genocide, or severe harassment likely to incite immediate harm.
     * Examples:
       * 'Burn every mosque to the ground.'
       * 'Every day when I see those fucking pride parades I want to beat up those faggots until I knock some sense into them.'
       * 'ENOUGH IS ENOUGH. We NEED to rid this country of those godforsaken Mexicans. We NEED to do whatever it takes.'

4. identity_targeted:
   * If hate_speech = 1, specify which identity group(s) are targeted. Choose from the following list (null if none):
     * 'women'
     * 'men'
     * 'Black'
     * 'White'
     * 'Jewish'
     * 'Muslim'
     * 'Christian'
     * 'Asian'
     * 'Latino/Hispanic'
     * 'LGBTQ+'
     * 'immigrants/refugees'
     * 'disabled people'
     * 'Indigenous'
     * 'other/unspecified'

Output Format:
Return your response as a valid JSON array containing objects for each ID. Each object must have these exact fields:

'id': (the ID number)
'hate_speech': (0 or 1)
'hate_speech_type': ('explicit', 'implicit', or null)
'hate_speech_severity': ('low', 'medium', 'high', or null)
'identity_targeted': (group name from list above or null)

Return ONLY the JSON array, no other text, no markdown formatting."""

# Few-shot examples for hate speech
hate_speech_few_shot = [
    {
        "user": """{
"tweet_id": 1823456712390884356,
"username": "RealTruthTeller",
"date": "2023-06-10 18:24:55+00:00",
"raw_content": "Immigrants are ruining our country. They take jobs, cause crime, and drain our economy. #facts"
}""",
        "assistant": """[
{"id": 1823456712390884356, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "medium", "identity_targeted": "immigrants/refugees"}
]"""
    },
    {
        "user": """{
"tweet_id": 1823460021399814723,
"username": "CivicMind",
"date": "2023-06-10 18:37:19+00:00",
"raw_content": "@RealTruthTeller That's just not true. Immigrants pay taxes, start small businesses, and make communities stronger."
}""",
        "assistant": """[
{"id": 1823460021399814723, "hate_speech": 0, "hate_speech_type": null, "hate_speech_severity": null, "identity_targeted": null}
]"""
    },
    {
        "user": """{
"tweet_id": 1823463812034739291,
"username": "FreedomGuy",
"date": "2023-06-10 18:54:28+00:00",
"raw_content": "@RealTruthTeller 100% agree. No one wants to say it but everyone knows it."
}""",
        "assistant": """[
{"id": 1823463812034739291, "hate_speech": 1, "hate_speech_type": "implicit", "hate_speech_severity": "low", "identity_targeted": "immigrants/refugees"}
]"""
    }
]

# ============================================================
# PROMPT 2: COUNTERSPEECH CLASSIFICATION
# ============================================================

counterspeech_system_prompt = """You will classify individual texts (replies) for counterspeech. Each text is paired with a unique Twitter/X ID and includes the parent text it's replying to.
Your job: return the counterspeech labels for each ID using the instructions below. Do not repeat the original text.

Counterspeech Classification
Fields to Add:

1. counterspeech:
   * 1 if the text challenges, refutes, condemns, or even mildly disagrees with hateful, discriminatory, or harmful content. 0 if otherwise.
   * Be liberal in detection: Even subtle disagreement, questioning, or redirection counts as counterspeech.
   * Look for: corrections, questioning premises, expressing discomfort, humor/sarcasm undermining hate, warning of consequences, expressing support for targets, empathetic language, or use hostile language to counter hate.
   * DO NOT require explicit condemnation - even mild pushback counts.
   * If the parent text contains hate speech, discriminatory views, or harmful stereotypes, and the reply challenges it, mark as counterspeech.

2. counterspeech_type:
   * If counterspeech = 1, choose the relevant categories out of the following (else null).

Category | Definition | Example
'presenting facts' | Correcting false or misleading statements with evidence. | 'Actually, refugee crime rates are lower than native-born citizens.'
'pointing out hypocrisy or contradictions' | Highlighting inconsistencies in the hate speaker's logic. Asking counter-questions. | 'You preach family values but cheat on your spouse — maybe rethink judging others.'
'warning of consequences' | Pointing out possible legal, social, or personal repercussions. | 'Inciting violence online can get you charged with a felony.'
'affiliation' | Establishing shared identity or solidarity with the target or both groups. | 'I'm a veteran, and I fought for everyone's freedom — including Muslims.'
'denouncing hate speech' | Explicitly condemning the hateful message. | 'This is blatant racism and it's disgusting.'
'humor' | Using jokes, irony, or sarcasm to undermine hate. | 'Wow, you've unlocked the 'Medieval Thinking' achievement badge.'
'empathy/positive tone' | Expressing compassion, kindness, or unity. | 'We're all trying to live decent lives — hate helps no one.'
'hostile language' | Responding to hate with insults or profanity while countering the message. | 'Only an idiot would believe that garbage.'

3. dominant_counterspeech_type:
   * If there is no counterspeech present, put null.
   * If there is one type of counterspeech present, put that type.
   * If there are multiple types present, choose the most salient and emphasized form.

IMPORTANT: Pay special attention to parent_hate_speech=1 cases. If the parent contains hate speech and the reply disagrees, questions, or challenges it in ANY way, mark as counterspeech=1.

Output Format:
Return your response as a valid JSON array containing objects for each ID. Each object must have these exact fields:

'id': (the ID number)
'counterspeech': (0 or 1)
'counterspeech_type': (one of the types listed above or null)
'dominant_counterspeech_type': (same as counterspeech_type if only one, or the dominant one if multiple, or null)

Return ONLY the JSON array, no other text, no markdown formatting."""

# Few-shot examples for counterspeech
counterspeech_few_shot = [
    {
        "user": """{
"tweet_id": 1823460021399814723,
"username": "CivicMind",
"date": "2023-06-10 18:37:19+00:00",
"raw_content": "@RealTruthTeller That's just not true. Immigrants pay taxes, start small businesses, and make communities stronger.",
"parent_tweet_id": 1823456712390884356,
"parent_raw_content": "Immigrants are ruining our country. They take jobs, cause crime, and drain our economy. #facts",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 1823460021399814723, "counterspeech": 1, "counterspeech_type": "presenting facts", "dominant_counterspeech_type": "presenting facts"}
]"""
    },
    {
        "user": """{
"tweet_id": 1823463812034739291,
"username": "FreedomGuy",
"date": "2023-06-10 18:54:28+00:00",
"raw_content": "@RealTruthTeller 100% agree. No one wants to say it but everyone knows it.",
"parent_tweet_id": 1823456712390884356,
"parent_raw_content": "Immigrants are ruining our country. They take jobs, cause crime, and drain our economy. #facts",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 1823463812034739291, "counterspeech": 0, "counterspeech_type": null, "dominant_counterspeech_type": null}
]"""
    },
    {
        "user": """{
"tweet_id": 1823470093440282623,
"username": "PeacefulMom",
"date": "2023-06-10 19:20:10+00:00",
"raw_content": "@RealTruthTeller Posts like this just divide people more. We all deserve respect, no matter where we're from ❤️",
"parent_tweet_id": 1823456712390884356,
"parent_raw_content": "Immigrants are ruining our country. They take jobs, cause crime, and drain our economy. #facts",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 1823470093440282623, "counterspeech": 1, "counterspeech_type": "empathy/positive tone", "dominant_counterspeech_type": "empathy/positive tone"}
]"""
    },
    {
        "user": """{
"tweet_id": 1823473049240911842,
"username": "User728392",
"date": "2023-06-10 19:32:52+00:00",
"raw_content": "@RealTruthTeller Not all immigrants are the same, man. Chill.",
"parent_tweet_id": 1823456712390884356,
"parent_raw_content": "Immigrants are ruining our country. They take jobs, cause crime, and drain our economy. #facts",
"parent_hate_speech": 1,
"parent_hate_speech_type": "implicit"
}""",
        "assistant": """[
{"id": 1823473049240911842, "counterspeech": 1, "counterspeech_type": "denouncing hate speech", "dominant_counterspeech_type": "denouncing hate speech"}
]"""
    },
    {
        "user": """{
"tweet_id": 1354976772333383680,
"username": "User123",
"date": "2021-01-28 20:15:30+00:00",
"raw_content": "like how are you going to call me a whore?? THAT'S THE WHOLE POINT. I AM!!",
"parent_tweet_id": 1354976700000000000,
"parent_raw_content": "if you can't handle my twitter and use it to insult me you will be blocked",
"parent_hate_speech": 0,
"parent_hate_speech_type": null
}""",
        "assistant": """[
{"id": 1354976772333383680, "counterspeech": 1, "counterspeech_type": "humor", "dominant_counterspeech_type": "humor"}
]"""
    },
    {
        "user": """{
"tweet_id": 1542205641787334656,
"username": "User456",
"date": "2022-06-29 18:45:12+00:00",
"raw_content": "@Kwite :( tf fuck i do",
"parent_tweet_id": 1542205600000000000,
"parent_raw_content": "Pro life more like I hate women",
"parent_hate_speech": 1,
"parent_hate_speech_type": "explicit"
}""",
        "assistant": """[
{"id": 1542205641787334656, "counterspeech": 1, "counterspeech_type": "denouncing hate speech", "dominant_counterspeech_type": "denouncing hate speech"}
]"""
    }
]


# ============================================================
# Helper functions
# ============================================================

def parse_model_output(raw_text):
    """Safely parse model JSON output into Python list."""
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        try:
            start = raw_text.find("[")
            end = raw_text.rfind("]") + 1
            return json.loads(raw_text[start:end])
        except Exception:
            return []

def classify_two_stage(batch, model_name):
    """
    Run hate speech classification first, then counterspeech classification.
    Returns combined predictions.
    Handles flexible field names for compatibility.
    
    IMPROVED: Now properly links parent-reply relationships and passes parent hate speech labels.
    """
    model = models[model_name]["model"]
    
    # Helper to normalize field names
    def normalize_item(item):
        """Convert various field name formats to standard format"""
        normalized = {}
        
        # ID field
        normalized['tweet_id'] = item.get('tweet_id') or item.get('id')
        
        # Username field
        normalized['username'] = item.get('username') or item.get('Username') or 'unknown'
        
        # Date field
        normalized['date'] = item.get('date') or item.get('Date') or 'unknown'
        
        # Content field
        normalized['raw_content'] = (
            item.get('raw_content') or 
            item.get('text') or 
            item.get('Text') or 
            item.get('content') or 
            ''
        )
        
        # Parent fields
        normalized['parent_tweet_id'] = (
            item.get('parent_tweet_id') or 
            item.get('parent_id') or 
            None
        )
        normalized['parent_raw_content'] = (
            item.get('parent_raw_content') or 
            item.get('parent_text') or 
            item.get('Parent_Text') or 
            ''
        )
        
        return normalized
    
    # Normalize all items in batch
    normalized_batch = [normalize_item(item) for item in batch]
    
    # Stage 1: Hate Speech Classification for ALL posts (including parent texts)
    # We need to classify both the posts AND their parents
    hate_messages = [
        {"role": "system", "content": hate_speech_system_prompt}
    ]
    for ex in hate_speech_few_shot:
        hate_messages.append({"role": "user", "content": ex["user"]})
        hate_messages.append({"role": "assistant", "content": ex["assistant"]})
    
    # Build list of texts to classify: both posts and their unique parents
    texts_to_classify = []
    parent_texts_map = {}  # Map parent text -> synthetic ID for classification
    
    for ex in normalized_batch:
        # Add the post itself
        texts_to_classify.append({
            "tweet_id": ex['tweet_id'],
            "username": ex['username'],
            "date": ex['date'],
            "raw_content": ex['raw_content']
        })
        
        # If it has a parent, add parent for classification too
        parent_text = ex.get('parent_raw_content', '').strip()
        if parent_text and parent_text not in parent_texts_map:
            # Create synthetic ID for parent (negative of post ID to avoid collisions)
            parent_id = -(ex['tweet_id'])
            parent_texts_map[parent_text] = parent_id
            texts_to_classify.append({
                "tweet_id": parent_id,
                "username": "parent_unknown",
                "date": ex['date'],
                "raw_content": parent_text
            })
    
    # Create hate speech classification prompt
    hate_items = []
    for text_entry in texts_to_classify:
        raw_content = (text_entry.get('raw_content') or '').replace('"', "'").replace('\n', ' ')
        hate_items.append({
            "tweet_id": text_entry['tweet_id'],
            "username": text_entry['username'],
            "date": text_entry['date'],
            "raw_content": raw_content
        })
    
    hate_user_content = json.dumps(hate_items, indent=2, ensure_ascii=False)
    hate_messages.append({"role": "user", "content": hate_user_content})
    
    # Query hate speech
    hate_response = model.query(messages=hate_messages)
    hate_raw = hate_response.choices[0].message.content
    hate_preds = parse_model_output(hate_raw)
    
    # Create lookup for hate speech results
    hate_lookup = {item["id"]: item for item in hate_preds if "id" in item}
    
    # Reverse map: parent text -> hate speech classification
    parent_hate_lookup = {}
    for parent_text, parent_id in parent_texts_map.items():
        if parent_id in hate_lookup:
            parent_hate_lookup[parent_text] = hate_lookup[parent_id]
    
    # Stage 2: Counterspeech Classification
    counter_messages = [
        {"role": "system", "content": counterspeech_system_prompt}
    ]
    for ex in counterspeech_few_shot:
        counter_messages.append({"role": "user", "content": ex["user"]})
        counter_messages.append({"role": "assistant", "content": ex["assistant"]})
    
    # Build counterspeech prompt with parent hate speech labels
    counter_items = []
    for ex in normalized_batch:
        parent_content = (ex.get('parent_raw_content') or '').replace('"', "'").replace('\n', ' ')
        raw_content = (ex.get('raw_content') or '').replace('"', "'").replace('\n', ' ')
        
        # Get parent hate speech classification if available
        parent_hate = parent_hate_lookup.get(ex.get('parent_raw_content', '').strip(), {})
        parent_hate_speech = parent_hate.get('hate_speech', 0)
        parent_hate_type = parent_hate.get('hate_speech_type', None)
        
        counter_item = {
            "tweet_id": ex['tweet_id'],
            "username": ex['username'],
            "date": ex['date'],
            "raw_content": raw_content,
            "parent_tweet_id": ex.get('parent_tweet_id'),
            "parent_raw_content": parent_content,
            "parent_hate_speech": parent_hate_speech,
            "parent_hate_speech_type": parent_hate_type
        }
        counter_items.append(counter_item)
    
    # Convert to JSON string
    counter_user_content = json.dumps(counter_items, indent=2, ensure_ascii=False)
    counter_messages.append({"role": "user", "content": counter_user_content})
    
    # Query counterspeech
    counter_response = model.query(messages=counter_messages)
    counter_raw = counter_response.choices[0].message.content
    counter_preds = parse_model_output(counter_raw)
    
    # Merge predictions (only include actual posts, not synthetic parent entries)
    merged_preds = {}
    for item in hate_preds:
        if "id" in item and item["id"] > 0:  # Only real posts (positive IDs)
            merged_preds[item["id"]] = item
    
    for item in counter_preds:
        if "id" in item:
            if item["id"] in merged_preds:
                merged_preds[item["id"]].update(item)
            else:
                merged_preds[item["id"]] = item
    
    return merged_preds, hate_raw, counter_raw


# Example usage for single test
if __name__ == "__main__":
    # Test with a small example using realistic Twitter data structure
    test_batch = [
        {
            "tweet_id": 1823456712390884356,
            "username": "RealTruthTeller",
            "date": "2023-06-10 18:24:55+00:00",
            "raw_content": "Immigrants are ruining our country. They take jobs, cause crime, and drain our economy. #facts",
            "parent_tweet_id": None,
            "parent_raw_content": None
        },
        {
            "tweet_id": 1823460021399814723,
            "username": "CivicMind",
            "date": "2023-06-10 18:37:19+00:00",
            "raw_content": "@RealTruthTeller That's just not true. Immigrants pay taxes, start small businesses, and make communities stronger.",
            "parent_tweet_id": 1823456712390884356,
            "parent_raw_content": "Immigrants are ruining our country. They take jobs, cause crime, and drain our economy. #facts"
        }
    ]
    
    print("\n=== Testing with gpt-5-mini ===")
    preds_mini, hate_raw_mini, counter_raw_mini = classify_two_stage(test_batch, "gpt-5-mini")
    print("\nHate Speech Output:")
    print(hate_raw_mini)
    print("\nCounterspeech Output:")
    print(counter_raw_mini)
    print("\nMerged Predictions:")
    print(json.dumps(preds_mini, indent=2))
    
    print("\n=== Testing with gpt-4o ===")
    preds_4o, hate_raw_4o, counter_raw_4o = classify_two_stage(test_batch, "gpt-4o")
    print("\nHate Speech Output:")
    print(hate_raw_4o)
    print("\nCounterspeech Output:")
    print(counter_raw_4o)
    print("\nMerged Predictions:")
    print(json.dumps(preds_4o, indent=2))


=== Testing with gpt-5-mini ===

Hate Speech Output:
[
  {
    "id": 1823456712390884356,
    "hate_speech": 1,
    "hate_speech_type": "implicit",
    "hate_speech_severity": "medium",
    "identity_targeted": "immigrants/refugees"
  },
  {
    "id": 1823460021399814723,
    "hate_speech": 0,
    "hate_speech_type": null,
    "hate_speech_severity": null,
    "identity_targeted": null
  },
  {
    "id": -1823460021399814723,
    "hate_speech": 1,
    "hate_speech_type": "implicit",
    "hate_speech_severity": "medium",
    "identity_targeted": "immigrants/refugees"
  }
]

Counterspeech Output:
[
  {
    "id": 1823456712390884356,
    "counterspeech": 0,
    "counterspeech_type": null,
    "dominant_counterspeech_type": null
  },
  {
    "id": 1823460021399814723,
    "counterspeech": 1,
    "counterspeech_type": "presenting facts",
    "dominant_counterspeech_type": "presenting facts"
  }
]

Merged Predictions:
{
  "1823456712390884356": {
    "id": 1823456712390884356,
    "hate_spe

In [9]:


#CELL 2

import time, json, numpy as np
from statistics import mean
from sklearn.metrics import accuracy_score, f1_score

# --- Load and merge data with labels ---
# Load the text data
with open("50_examples_converted.json", "r", encoding="utf-8") as f:
    text_data = {int(d["ID"]): d for d in json.load(f)}

# Load the labels
with open("50_annotated_examples.json", "r", encoding="utf-8") as f:
    labels = {int(d["id"]): d for d in json.load(f)}

# Merge text data with labels to create complete ground truth
ground_truth = {}
for id_val in labels.keys():
    if id_val in text_data:
        # Merge both dictionaries
        ground_truth[id_val] = {**text_data[id_val], **labels[id_val]}
    else:
        print(f"Warning: ID {id_val} has labels but no text data")

print(f"Loaded {len(ground_truth)} examples with text and labels")

# --- Prepare data for classification ---
# Convert ground truth to list format for classification
examples = []
for id_val, data in ground_truth.items():
    example = {
        'tweet_id': data.get('ID'),
        'id': data.get('ID'),  # Keep both for compatibility
        'username': data.get('Username', 'unknown'),
        'date': data.get('Date', 'unknown'),
        'raw_content': data.get('Text', ''),
        'text': data.get('Text', ''),  # Keep both for compatibility
        'parent_raw_content': data.get('Parent_Text'),
        'parent_text': data.get('Parent_Text'),  # Keep both for compatibility
        'parent_tweet_id': None  # Will be set if we can find parent in dataset
    }
    examples.append(example)

print(f"Prepared {len(examples)} examples for classification")
print(f"Sample example keys: {list(examples[0].keys())}")

# Remaining code continues as before
batch_sizes = [50]
num_iterations = 1
model_names = ["gpt-5-mini", "gpt-4o"]
results_summary = []

# ==============================================================
# Helper functions
# ==============================================================

def parse_model_output(raw_text):
    """Safely parse model JSON output into Python list."""
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        try:
            start = raw_text.find("[")
            end = raw_text.rfind("]") + 1
            return json.loads(raw_text[start:end])
        except Exception:
            return []

# ==============================================================
# Metric computation
# ==============================================================

def compute_metrics(preds, gold):
    """Compute metrics for binary, multiclass, and multilabel fields."""

    # --- Helper normalizers ---
    def safe_label(val):
        if not val or str(val).lower() in {"unknown", "none", "null", "nan"}:
            return "none"
        return str(val).strip()

    def normalize_identities(x):
        if x is None:
            return []
        if isinstance(x, str):
            return [x]
        if isinstance(x, list):
            return x
        return []

    # --- Binary fields ---
    y_true_hate, y_pred_hate = [], []
    y_true_counter, y_pred_counter = [], []

    # --- Multiclass fields ---
    y_true_type, y_pred_type = [], []
    y_true_severity, y_pred_severity = [], []
    y_true_counter_type, y_pred_counter_type = [], []

    # --- Multilabel (identities) ---
    base_identities = [
        'women', 'men', 'Black', 'White', 'Jewish', 'Muslim', 'Christian',
        'Asian', 'Latino/Hispanic', 'LGBTQ+', 'immigrants/refugees',
        'disabled people', 'Indigenous', 'other/unspecified'
    ]
    all_identities = set(base_identities)

    # Dynamically extend identity list with unseen ones
    for d in gold.values():
        for x in normalize_identities(d.get("identity_targeted")):
            all_identities.add(x)
    for p in preds.values():
        for x in normalize_identities(p.get("identity_targeted")):
            all_identities.add(x)
    all_identities = sorted(all_identities)

    y_true_id, y_pred_id = [], []

    # --- Collect gold and predicted values ---
    for pid, p in preds.items():
        g = gold.get(pid)
        if not g:
            continue

        # Binary
        y_true_hate.append(g["hate_speech"])
        y_pred_hate.append(p.get("hate_speech", 0))
        y_true_counter.append(g["counterspeech"])
        y_pred_counter.append(p.get("counterspeech", 0))

        # Multiclass (cleaned)
        y_true_type.append(safe_label(g.get("hate_speech_type")))
        y_pred_type.append(safe_label(p.get("hate_speech_type")))
        y_true_severity.append(safe_label(g.get("hate_speech_severity")))
        y_pred_severity.append(safe_label(p.get("hate_speech_severity")))
        y_true_counter_type.append(safe_label(g.get("counterspeech_type")))
        y_pred_counter_type.append(safe_label(p.get("counterspeech_type")))

        # Multilabel (normalized)
        gt_raw = normalize_identities(g.get("identity_targeted"))
        pr_raw = normalize_identities(p.get("identity_targeted"))
        gt_set, pr_set = set(gt_raw), set(pr_raw)
        y_true_id.append([1 if i in gt_set else 0 for i in all_identities])
        y_pred_id.append([1 if i in pr_set else 0 for i in all_identities])

    # --- Compute metrics safely ---
    metrics = {
        # Binary
        "hate_acc": round(accuracy_score(y_true_hate, y_pred_hate), 3),
        "hate_f1": round(f1_score(y_true_hate, y_pred_hate), 3),
        "counter_acc": round(accuracy_score(y_true_counter, y_pred_counter), 3),
        "counter_f1": round(f1_score(y_true_counter, y_pred_counter), 3),

        # Multiclass (macro F1 averages over all categories)
        "type_acc": round(accuracy_score(y_true_type, y_pred_type), 3),
        "type_f1": round(f1_score(y_true_type, y_pred_type, average="macro"), 3),

        "severity_acc": round(accuracy_score(y_true_severity, y_pred_severity), 3),
        "severity_f1": round(f1_score(y_true_severity, y_pred_severity, average="macro"), 3),

        "counter_type_acc": round(accuracy_score(y_true_counter_type, y_pred_counter_type), 3),
        "counter_type_f1": round(f1_score(y_true_counter_type, y_pred_counter_type, average="macro"), 3),

        # Multilabel (micro average)
        "identity_acc": round(accuracy_score(np.array(y_true_id).flatten(),
                                             np.array(y_pred_id).flatten()), 3),
        "identity_f1": round(f1_score(y_true_id, y_pred_id, average="micro"), 3)
    }

    return metrics

# ==============================================================
# Run experiments comparing both models
# ==============================================================

for model_name in model_names:
    print(f"\n{'='*60}")
    print(f"Testing model: {model_name}")
    print(f"{'='*60}")
    
    for batch_size in batch_sizes:
        print(f"\n=== Testing batch size {batch_size} for {num_iterations} iterations ===")
        all_iter_results = []
        all_iter_times = []

        for iteration in range(num_iterations):
            print(f"Iteration {iteration + 1}/{num_iterations}")
            batch_times, preds = [], {}

            for i in range(0, len(examples), batch_size):
                batch = examples[i:i + batch_size]
                start = time.time()
                merged_preds, _, _ = classify_two_stage(batch, model_name)
                elapsed = time.time() - start
                batch_times.append(elapsed)
                preds.update(merged_preds)

                # ✅ Pydantic v2 compatible model stats logging
                with open(f"usage_log_{model_name}.json", "w") as f:
                    json.dump(models[model_name]["model"].stats.model_dump(), f, indent=2)
    
            # --- Record metrics for this iteration ---
            metrics = compute_metrics(preds, ground_truth)
            total_time, avg_time = sum(batch_times), mean(batch_times)
            all_iter_times.append(avg_time)
            all_iter_results.append(metrics)

        # --- Aggregate over iterations ---
        avg_metrics = {
            "hate_acc": round(mean([m["hate_acc"] for m in all_iter_results]), 3),
            "hate_f1": round(mean([m["hate_f1"] for m in all_iter_results]), 3),
            "counter_acc": round(mean([m["counter_acc"] for m in all_iter_results]), 3),
            "counter_f1": round(mean([m["counter_f1"] for m in all_iter_results]), 3),
            "type_f1": round(mean([m["type_f1"] for m in all_iter_results]), 3),
            "severity_f1": round(mean([m["severity_f1"] for m in all_iter_results]), 3),
            "counter_type_f1": round(mean([m["counter_type_f1"] for m in all_iter_results]), 3),
            "identity_f1": round(mean([m["identity_f1"] for m in all_iter_results]), 3),
        }

        results_summary.append({
            "model": model_name,
            "batch_size": batch_size,
            "iterations": num_iterations,
            "avg_time_s": round(mean(all_iter_times), 2),
            **avg_metrics,
        })

# ==============================================================
# Final Summary with Model Comparison
# ==============================================================

print("\n" + "="*80)
print("FINAL COMPARISON: GPT-5-mini vs GPT-4o")
print("="*80)
for r in results_summary:
    print(f"\nModel: {r['model']}")
    print(f"  Batch Size: {r['batch_size']}")
    print(f"  Avg Time: {r['avg_time_s']}s")
    print(f"  Hate Speech - Acc: {r['hate_acc']}, F1: {r['hate_f1']}")
    print(f"  Counterspeech - Acc: {r['counter_acc']}, F1: {r['counter_f1']}")
    print(f"  Type F1: {r['type_f1']}, Severity F1: {r['severity_f1']}")
    print(f"  Counter Type F1: {r['counter_type_f1']}, Identity F1: {r['identity_f1']}")


Loaded 50 examples with text and labels
Prepared 50 examples for classification
Sample example keys: ['tweet_id', 'id', 'username', 'date', 'raw_content', 'text', 'parent_raw_content', 'parent_text', 'parent_tweet_id']

Testing model: gpt-5-mini

=== Testing batch size 50 for 1 iterations ===
Iteration 1/1

Testing model: gpt-4o

=== Testing batch size 50 for 1 iterations ===
Iteration 1/1

FINAL COMPARISON: GPT-5-mini vs GPT-4o

Model: gpt-5-mini
  Batch Size: 50
  Avg Time: 293.73s
  Hate Speech - Acc: 0.84, F1: 0.636
  Counterspeech - Acc: 0.84, F1: 0.429
  Type F1: 0.599, Severity F1: 0.304
  Counter Type F1: 0.272, Identity F1: 0.455

Model: gpt-4o
  Batch Size: 50
  Avg Time: 30.69s
  Hate Speech - Acc: 0.84, F1: 0.636
  Counterspeech - Acc: 0.78, F1: 0.421
  Type F1: 0.542, Severity F1: 0.228
  Counter Type F1: 0.275, Identity F1: 0.364


In [10]:

#Cell 3

import time, json

# Use the same data preparation as Cell 2
with open("50_examples_converted.json", "r", encoding="utf-8") as f:
    text_data = {int(d["ID"]): d for d in json.load(f)}

with open("50_annotated_examples.json", "r", encoding="utf-8") as f:
    labels = {int(d["id"]): d for d in json.load(f)}

ground_truth = {}
for id_val in labels.keys():
    if id_val in text_data:
        ground_truth[id_val] = {**text_data[id_val], **labels[id_val]}

examples = []
for id_val, data in ground_truth.items():
    example = {
        'tweet_id': data.get('ID'),
        'id': data.get('ID'),
        'username': data.get('Username', 'unknown'),
        'date': data.get('Date', 'unknown'),
        'raw_content': data.get('Text', ''),
        'text': data.get('Text', ''),
        'parent_raw_content': data.get('Parent_Text'),
        'parent_text': data.get('Parent_Text'),
        'parent_tweet_id': None
    }
    examples.append(example)

# --- Run test with both models for batch size 50 ---
batch_size = 50

for model_name in model_names:
    print(f"\n{'='*60}")
    print(f"Running single batch test with {model_name} (batch size {batch_size})")
    print(f"{'='*60}")
    
    preds = {}
    all_outputs = []  # raw model responses (for debugging)
    batch_times = []
    
    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]

        start = time.time()
        merged_preds, hate_raw, counter_raw = classify_two_stage(batch, model_name)
        elapsed = time.time() - start
        batch_times.append(elapsed)

        all_outputs.append({
            "batch_index": i // batch_size,
            "time_s": round(elapsed, 2),
            "hate_raw_output": hate_raw,
            "counter_raw_output": counter_raw
        })

        preds.update(merged_preds)

    print(f"✅ Completed batch of size {batch_size} in {round(sum(batch_times), 2)}s")

    # --- Save outputs ---
    with open(f"model_raw_outputs_batch50_{model_name}.json", "w", encoding="utf-8") as f:
        json.dump(all_outputs, f, indent=2, ensure_ascii=False)

    with open(f"model_parsed_preds_batch50_{model_name}.json", "w", encoding="utf-8") as f:
        json.dump(preds, f, indent=2, ensure_ascii=False)

    print(f"💾 Saved for {model_name}:")
    print(f"  - model_raw_outputs_batch50_{model_name}.json (raw model responses)")
    print(f"  - model_parsed_preds_batch50_{model_name}.json (parsed per-post predictions)")

print("\n" + "="*80)
print("All tests completed! Check output files for detailed results.")
print("="*80)


Running single batch test with gpt-5-mini (batch size 50)
✅ Completed batch of size 50 in 271.23s
💾 Saved for gpt-5-mini:
  - model_raw_outputs_batch50_gpt-5-mini.json (raw model responses)
  - model_parsed_preds_batch50_gpt-5-mini.json (parsed per-post predictions)

Running single batch test with gpt-4o (batch size 50)
✅ Completed batch of size 50 in 35.94s
💾 Saved for gpt-4o:
  - model_raw_outputs_batch50_gpt-4o.json (raw model responses)
  - model_parsed_preds_batch50_gpt-4o.json (parsed per-post predictions)

All tests completed! Check output files for detailed results.


In [11]:
import json

print("="*80)
print("INSPECTING RAW MODEL OUTPUTS")
print("="*80)

# Load raw outputs
with open("model_raw_outputs_batch50_gpt-5-mini.json", "r") as f:
    raw_mini = json.load(f)

print("\n" + "="*60)
print("GPT-5-MINI RAW OUTPUTS")
print("="*60)

print("\n--- HATE SPEECH OUTPUT (first 1000 chars) ---")
print(raw_mini[0]["hate_raw_output"][:1000])

print("\n--- COUNTERSPEECH OUTPUT (full) ---")
print(raw_mini[0]["counter_raw_output"])

# Load predictions
with open("model_parsed_preds_batch50_gpt-5-mini.json", "r") as f:
    preds = json.load(f)

print("\n" + "="*60)
print("PARSED PREDICTIONS SAMPLE")
print("="*60)

# Show first 5 predictions
count = 0
for id_val, pred in list(preds.items())[:5]:
    count += 1
    print(f"\n{count}. ID: {id_val}")
    print(f"   Counterspeech: {pred.get('counterspeech', 'MISSING')}")
    print(f"   Counter Type: {pred.get('counterspeech_type', 'MISSING')}")
    print(f"   All keys: {list(pred.keys())}")

# Load ground truth
with open("50_examples_converted.json", "r") as f:
    text_data = {int(d["ID"]): d for d in json.load(f)}

with open("50_annotated_examples.json", "r") as f:
    labels = {int(d["id"]): d for d in json.load(f)}

print("\n" + "="*60)
print("GROUND TRUTH FOR COUNTERSPEECH=1 EXAMPLES")
print("="*60)

counter_examples = [(id_val, data) for id_val, data in labels.items() if data.get("counterspeech") == 1]

for i, (id_val, label_data) in enumerate(counter_examples[:3]):
    text_entry = text_data.get(id_val, {})
    print(f"\n{i+1}. ID: {id_val}")
    print(f"   Text: {text_entry.get('Text', 'N/A')[:100]}")
    print(f"   Parent: {text_entry.get('Parent_Text', 'N/A')[:100]}")
    print(f"   Counter Type (GT): {label_data.get('counterspeech_type')}")
    print(f"   Predicted Counter: {preds.get(str(id_val), {}).get('counterspeech', 'NOT IN PREDS')}")

print("\n" + "="*80)
print("DIAGNOSIS")
print("="*80)

# Check if counterspeech field is even in predictions
has_counter = sum(1 for p in preds.values() if 'counterspeech' in p)
counter_ones = sum(1 for p in preds.values() if p.get('counterspeech') == 1)

print(f"\nTotal predictions: {len(preds)}")
print(f"Predictions with 'counterspeech' field: {has_counter}")
print(f"Predictions with counterspeech=1: {counter_ones}")
print(f"Predictions with counterspeech=0: {has_counter - counter_ones}")

if counter_ones == 0:
    print("\n⚠️ PROBLEM: Model is predicting counterspeech=0 for ALL examples")
    print("\nPossible issues:")
    print("1. Counterspeech prompt is too strict")
    print("2. Parent hate speech labels not being properly passed")
    print("3. Model can't see the parent-child relationship")
    print("4. Raw output is empty or malformed")

INSPECTING RAW MODEL OUTPUTS

GPT-5-MINI RAW OUTPUTS

--- HATE SPEECH OUTPUT (first 1000 chars) ---
[
{"id":1239062934799536128,"hate_speech":0,"hate_speech_type":null,"hate_speech_severity":null,"identity_targeted":null},
{"id":-1239062934799536128,"hate_speech":1,"hate_speech_type":"implicit","hate_speech_severity":"medium","identity_targeted":"Asian"},
{"id":1239063402569322496,"hate_speech":1,"hate_speech_type":"explicit","hate_speech_severity":"medium","identity_targeted":"Asian"},
{"id":-1239063402569322496,"hate_speech":1,"hate_speech_type":"explicit","hate_speech_severity":"medium","identity_targeted":"Asian"},
{"id":1542205641787334656,"hate_speech":0,"hate_speech_type":null,"hate_speech_severity":null,"identity_targeted":null},
{"id":-1542205641787334656,"hate_speech":1,"hate_speech_type":"explicit","hate_speech_severity":"medium","identity_targeted":"women"},
{"id":1291456770293071872,"hate_speech":1,"hate_speech_type":"explicit","hate_speech_severity":"low","identity_target

In [12]:
import time, json

# --- Load annotated ground truth ---
with open("50_annotated_examples.json", "r", encoding="utf-8") as f:
    ground_truth = {int(d["id"]): d for d in json.load(f)}

# --- Prepare data (using your parser + few-shot messages setup) ---
examples = list(parser.iter_items())
batch_size = 50

# --- Helper functions ---
def build_messages(batch):
    """Helper to construct per-batch messages for model query."""
    return [
        messages[0],         # system prompt
        *messages[1:-1],     # few-shot examples
        {
            "role": "user",
            "content": "\n\n".join(
                [f"ID: {ex['id']}\nText: {ex['text']}\nParent_Text: {ex['parent_text']}" for ex in batch]
            ),
        },
    ]

def parse_model_output(raw_text):
    """Safely parse model JSON output into Python list."""
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        try:
            start = raw_text.find("[")
            end = raw_text.rfind("]") + 1
            return json.loads(raw_text[start:end])
        except Exception:
            return []

# --- Run single batch test ---
preds = {}
all_outputs = []  # stores raw responses and timings
batch_times = []

print(f"\n=== Running single batch test with batch size {batch_size} ===")

for i in range(0, len(examples), batch_size):
    batch = examples[i:i + batch_size]
    batched_messages = build_messages(batch)

    start = time.time()
    resp = model.query(messages=batched_messages)
    elapsed = time.time() - start
    batch_times.append(elapsed)

    raw_text = resp.choices[0].message.content
    all_outputs.append({
        "batch_index": i // batch_size,
        "time_s": round(elapsed, 2),
        "raw_output": raw_text
    })

    parsed = parse_model_output(raw_text)
    for item in parsed:
        if "id" in item:
            preds[int(item["id"])] = item

print(f"✅ Completed batch of size {batch_size} in {round(sum(batch_times), 2)}s")

# --- Save results ---
with open("model_raw_outputs_batch50.json", "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=2, ensure_ascii=False)

with open("model_parsed_preds_batch50.json", "w", encoding="utf-8") as f:
    json.dump(preds, f, indent=2, ensure_ascii=False)

print("💾 Saved:")
print("  - model_raw_outputs_batch50.json (raw GPT responses)")
print("  - model_parsed_preds_batch50.json (parsed predictions per post)")



=== Running single batch test with batch size 50 ===


NameError: name 'messages' is not defined

In [ ]:
'''
# === Batch Size + Ground Truth Evaluation ===
import time, json, numpy as np
from statistics import mean
from sklearn.metrics import accuracy_score, f1_score

# --- Load ground truth labels ---
with open("50_annotated_examples.json", "r", encoding="utf-8") as f:
    ground_truth = {int(d["id"]): d for d in json.load(f)}

# --- Prepare data ---
examples = list(parser.iter_items())
batch_sizes = [32, 50]
results_summary = []

def build_messages(batch):
    """Helper to construct per-batch messages for model query."""
    return [
        messages[0],         # system prompt
        *messages[1:-1],     # few-shot examples
        {
            "role": "user",
            "content": "\n\n".join(
                [f"ID: {ex['id']}\nText: {ex['text']}\nParent_Text: {ex['parent_text']}" for ex in batch]
            ),
        },
    ]

def parse_model_output(raw_text):
    """Safely parse model JSON output into Python list."""
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        # Try to recover from leading/trailing text
        try:
            start = raw_text.find("[")
            end = raw_text.rfind("]") + 1
            return json.loads(raw_text[start:end])
        except Exception:
            return []

def compute_metrics(preds, gold):
    """Compute accuracy & F1 for hate_speech and counterspeech."""
    y_true_hate, y_pred_hate = [], []
    y_true_counter, y_pred_counter = [], []
    for pid, p in preds.items():
        g = gold.get(pid)
        if not g:
            continue
        y_true_hate.append(g["hate_speech"])
        y_pred_hate.append(p.get("hate_speech", 0))
        y_true_counter.append(g["counterspeech"])
        y_pred_counter.append(p.get("counterspeech", 0))
    return {
        "hate_acc": round(accuracy_score(y_true_hate, y_pred_hate), 3),
        "hate_f1": round(f1_score(y_true_hate, y_pred_hate), 3),
        "counter_acc": round(accuracy_score(y_true_counter, y_pred_counter), 3),
        "counter_f1": round(f1_score(y_true_counter, y_pred_counter), 3),
    }

# --- Run experiments ---
for batch_size in batch_sizes:
    print(f"\n=== Testing batch size {batch_size} ===")
    batch_times, preds = [], {}

    for i in range(0, len(examples), batch_size):
        batch = examples[i:i + batch_size]
        batched_messages = build_messages(batch)
        start = time.time()
        resp = model.query(messages=batched_messages)
        elapsed = time.time() - start
        batch_times.append(elapsed)

        raw_text = resp.choices[0].message.content
        parsed = parse_model_output(raw_text)
        for item in parsed:
            if "id" in item:
                preds[int(item["id"])] = item

    with open("usage_log.json", "w") as f:
        json.dump(model.stats.dict(), f, indent=2)

    metrics = compute_metrics(preds, ground_truth)
    total_time, avg_time = sum(batch_times), mean(batch_times)
    results_summary.append({
        "batch_size": batch_size,
        "batches": len(batch_times),
        "total_time_s": round(total_time, 2),
        "avg_time_s": round(avg_time, 2),
        **metrics,
    })

# --- Display summary ---
print("\n=== Batch Size Evaluation Summary ===")
for r in results_summary:
    print(r)
'''

'\n# === Batch Size + Ground Truth Evaluation ===\nimport time, json, numpy as np\nfrom statistics import mean\nfrom sklearn.metrics import accuracy_score, f1_score\n\n# --- Load ground truth labels ---\nwith open("50_annotated_examples.json", "r", encoding="utf-8") as f:\n    ground_truth = {int(d["id"]): d for d in json.load(f)}\n\n# --- Prepare data ---\nexamples = list(parser.iter_items())\nbatch_sizes = [32, 50]\nresults_summary = []\n\ndef build_messages(batch):\n    """Helper to construct per-batch messages for model query."""\n    return [\n        messages[0],         # system prompt\n        *messages[1:-1],     # few-shot examples\n        {\n            "role": "user",\n            "content": "\n\n".join(\n                [f"ID: {ex[\'id\']}\nText: {ex[\'text\']}\nParent_Text: {ex[\'parent_text\']}" for ex in batch]\n            ),\n        },\n    ]\n\ndef parse_model_output(raw_text):\n    """Safely parse model JSON output into Python list."""\n    try:\n        return js

In [ ]:
# Print the full raw response object (optional, for debugging)
print(completion)

# Print just the text output
print("\n=== MODEL RESPONSE ===\n")
print(completion.choices[0].message["content"])

# If you want *all* responses (sometimes n > 1):
for i, choice in enumerate(completion.choices, start=1):
    print(f"\n=== RESPONSE {i} ===\n")
    print(choice.message["content"])

NameError: name 'completion' is not defined

# Usage with vLLM
- Install vLLM and launch the server. It will give you a host url.
- Example command: `CUDA_VISIBLE_DEVICES=0 vllm serve Qwen/Qwen3-1.7B --trust-remote-code`
- Running this command should give you a host url. You can use it as follows.

In [ ]:
# In your script, import the APIModelConfig and LiteLLMModel classes
config = APIModelConfig(model_name="hosted_vllm/Qwen3-1.7B", temperature=0.7, top_p=1.0, max_tokens=1000, host_url="http://0.0.0.0:8000")
model = LiteLLMModel(config=config, name="hosted_vllm/Qwen3-1.7B", logger=logging.getLogger(__name__))

# Prepare the messages
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of the moon?"},
]

# Call the model
response = model.query(messages=messages)
print(response.choices[0].message.content)


Using custom host URL: http://0.0.0.0:8000. Cost management will not be available. Register the model with litellm to enable cost management. See https://docs.litellm.ai/docs/completion/token_usage#9-register_model
Error querying hosted_vllm/Qwen3-1.7B with tools
Traceback (most recent call last):
  File "/data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages/litellm/llms/openai/openai.py", line 736, in completion
    raise e
  File "/data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages/litellm/llms/openai/openai.py", line 664, in completion
    ) = self.make_sync_openai_chat_completion_request(
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages/litellm/litellm_core_utils/logging_utils.py", line 237, in sync_wrapper
    result = func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/data4/anika/UCSBCounterspeechProject/.venv/lib/python3.11/site-packages/l


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



NotFoundError: litellm.NotFoundError: NotFoundError: Hosted_vllmException - Error code: 404 - {'detail': 'Not Found'}